# Черновая исследовательская тетрадь

В этой Colab-ориентированной тетради собраны основные версии этапов проекта: графовая прямая модель, классификационные эксперименты, регрессионное восстановление параметров и последующее уточнение.

**Статус:** исследовательский код в разработке. Результаты рассчитаны на синтетических данных; это не проверка на полевых измерениях. Сохранённые выводы удалены при подготовке репозитория, а код этапов оставлен.

Запускай этапы последовательно в Google Colab. Некоторые ячейки используют предыдущие файлы, сохраняют материалы в `/content` и содержат вспомогательные функции скачивания результатов. Зависимости перечислены в `requirements.txt`.


## Этап 0 — графовая прямая модель

Итоговый Colab-ready скрипт для первой недели проекта:
прямая задача геометрической оптики для простой слоистой модели.

Что строит программа:
1. Простую слоистую модель среды.
2. Лучи минимального времени от точечного источника до приемников.
3. Кривую времени прихода волны на приемники.
4. Дополнительную карту времен прихода.

Физические ограничения:
- скорость в слоях задается в м/с;
- допустимый диапазон: 1000–10000 м/с, то есть 1–10 км/с;
- в нижних слоях скорость должна быть больше или равна скорости в верхних слоях.

Запуск в Google Colab:
1. Открой новый ноутбук.
2. Загрузи этот файл или вставь код в одну ячейку.
3. Запусти.
4. В результате появятся картинки:
   - layered_model_rays_and_times.png
   - layered_time_map.png

In [ ]:
import heapq
import math
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np


# ============================================================
# 1. НАСТРОЙКИ МОДЕЛИ
# ============================================================

@dataclass
class ModelConfig:
    # Размер расчетной области.
    # x — горизонтальная координата, метры.
    # z — глубина, метры. z = 0 — поверхность.
    x_min: float = 0.0
    x_max: float = 1200.0
    z_min: float = 0.0
    z_max: float = 500.0

    # Количество точек сетки.
    # Чем больше nx и nz, тем точнее расчет, но тем дольше работает программа.
    nx: int = 241
    nz: int = 101

    # Координаты точечного источника.
    source_x: float = 600.0
    source_z: float = 150.0

    # Приемники стоят на поверхности.
    receiver_count: int = 25
    receiver_z: float = 0.0


# Простая слоистая модель.
# Формат каждого слоя:
# (верхняя_граница_глубины, нижняя_граница_глубины, скорость_м_с)

# Важно:
# - скорости должны быть от 1000 до 10000 м/с;
# - скорость в нижних слоях должна быть не меньше, чем в верхних.
LAYERS = [
    (0.0, 120.0, 1500.0),
    (120.0, 300.0, 2200.0),
    (300.0, 500.0, 3200.0),
]


# ============================================================
# 2. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================

def build_grid(config: ModelConfig):
    """
    Создает расчетную сетку по x и z.
    """
    xs = np.linspace(config.x_min, config.x_max, config.nx)
    zs = np.linspace(config.z_min, config.z_max, config.nz)

    dx = xs[1] - xs[0]
    dz = zs[1] - zs[0]

    return xs, zs, dx, dz


def nearest_index(array: np.ndarray, value: float) -> int:
    """
    Возвращает индекс ближайшей точки сетки к заданной координате.
    """
    return int(np.argmin(np.abs(array - value)))


def validate_layers(layers):
    """
    Проверяет физические ограничения:
    1. Скорости лежат в диапазоне 1–10 км/с.
    2. В нижних слоях скорость не меньше, чем в верхних.
    """
    previous_velocity = -np.inf

    for layer_number, (z_top, z_bottom, velocity) in enumerate(layers, start=1):
        if z_bottom <= z_top:
            raise ValueError(
                f"Ошибка в слое {layer_number}: нижняя граница должна быть больше верхней."
            )

        if not (1000.0 <= velocity <= 10000.0):
            raise ValueError(
                f"Ошибка в слое {layer_number}: скорость {velocity} м/с вне диапазона 1000–10000 м/с."
            )

        if velocity < previous_velocity:
            raise ValueError(
                f"Ошибка в слое {layer_number}: скорость в нижнем слое меньше, чем выше."
            )

        previous_velocity = velocity


def velocity_by_depth(z: float, layers=LAYERS) -> float:
    """
    Возвращает скорость волны на заданной глубине z.
    """
    for z_top, z_bottom, velocity in layers:
        if z_top <= z <= z_bottom:
            return velocity

    # Если точка случайно вышла за список слоев,
    # берем скорость последнего слоя.
    return layers[-1][2]


def build_velocity_model(xs: np.ndarray, zs: np.ndarray, layers=LAYERS) -> np.ndarray:
    """
    Создает 2D-карту скоростей.

    В простой слоистой модели скорость зависит только от глубины z,
    поэтому по горизонтали внутри одного слоя она одинаковая.
    """
    velocity_model = np.zeros((len(zs), len(xs)), dtype=float)

    for iz, z in enumerate(zs):
        velocity_model[iz, :] = velocity_by_depth(z, layers)

    return velocity_model


# ============================================================
# 3. РАСЧЕТ ВРЕМЕН АЛГОРИТМОМ ДЕЙКСТРЫ
# ============================================================

def dijkstra_with_paths(
    velocity_model: np.ndarray,
    dx: float,
    dz: float,
    source_iz: int,
    source_ix: int
):
    """
    Считает минимальные времена прихода волны от источника до всех точек сетки.

    Идея:
    - каждая точка сетки — вершина графа;
    - из каждой точки можно перейти к соседним точкам;
    - стоимость перехода равна времени:
          время = расстояние / средняя_скорость;
    - алгоритм Дейкстры находит путь с минимальным временем.

    Это дискретное приближение идеи Fast Marching Method.
    """
    nz, nx = velocity_model.shape

    travel_times = np.full((nz, nx), np.inf, dtype=float)

    # parent_iz и parent_ix нужны, чтобы потом восстановить траектории лучей.
    parent_iz = np.full((nz, nx), -1, dtype=int)
    parent_ix = np.full((nz, nx), -1, dtype=int)

    # 8 направлений движения по сетке:
    # вверх, вниз, влево, вправо и 4 диагонали.
    neighbors = [
        (-1, 0), (1, 0), (0, -1), (0, 1),
        (-1, -1), (-1, 1), (1, -1), (1, 1),
    ]

    travel_times[source_iz, source_ix] = 0.0
    heap = [(0.0, source_iz, source_ix)]

    while heap:
        current_time, iz, ix = heapq.heappop(heap)

        # Если это устаревшая запись, пропускаем ее.
        if current_time > travel_times[iz, ix]:
            continue

        for dzi, dxi in neighbors:
            next_iz = iz + dzi
            next_ix = ix + dxi

            if not (0 <= next_iz < nz and 0 <= next_ix < nx):
                continue

            step_distance = math.sqrt((dxi * dx) ** 2 + (dzi * dz) ** 2)

            average_velocity = 0.5 * (
                velocity_model[iz, ix] + velocity_model[next_iz, next_ix]
            )

            step_time = step_distance / average_velocity
            new_time = current_time + step_time

            if new_time < travel_times[next_iz, next_ix]:
                travel_times[next_iz, next_ix] = new_time

                parent_iz[next_iz, next_ix] = iz
                parent_ix[next_iz, next_ix] = ix

                heapq.heappush(heap, (new_time, next_iz, next_ix))

    return travel_times, parent_iz, parent_ix


def reconstruct_path(target_iz: int, target_ix: int, parent_iz: np.ndarray, parent_ix: np.ndarray):
    """
    Восстанавливает путь минимального времени от источника до выбранной точки.
    """
    path = []
    iz, ix = int(target_iz), int(target_ix)

    while iz != -1 and ix != -1:
        path.append((iz, ix))

        previous_iz = parent_iz[iz, ix]
        previous_ix = parent_ix[iz, ix]

        if previous_iz == -1 and previous_ix == -1:
            break

        iz, ix = previous_iz, previous_ix

    path.reverse()
    return path


# ============================================================
# 4. ПРИЕМНИКИ И ВРЕМЕНА НА ПОВЕРХНОСТИ
# ============================================================

def get_receiver_coordinates(xs: np.ndarray, config: ModelConfig):
    """
    Создает координаты приемников на поверхности.
    """
    margin = 50.0

    receiver_xs = np.linspace(
        config.x_min + margin,
        config.x_max - margin,
        config.receiver_count
    )

    return receiver_xs


def get_receiver_times_and_paths(
    xs: np.ndarray,
    zs: np.ndarray,
    travel_times: np.ndarray,
    parent_iz: np.ndarray,
    parent_ix: np.ndarray,
    config: ModelConfig
):
    """
    Для каждого приемника:
    - берет время прихода волны;
    - восстанавливает луч минимального времени.
    """
    receiver_xs = get_receiver_coordinates(xs, config)

    receiver_iz = nearest_index(zs, config.receiver_z)

    receiver_times = []
    ray_paths = []

    for receiver_x in receiver_xs:
        receiver_ix = nearest_index(xs, receiver_x)

        receiver_times.append(travel_times[receiver_iz, receiver_ix])

        path = reconstruct_path(receiver_iz, receiver_ix, parent_iz, parent_ix)
        ray_paths.append(path)

    return receiver_xs, np.array(receiver_times), ray_paths


# ============================================================
# 5. ПОСТРОЕНИЕ ГРАФИКОВ
# ============================================================

def plot_results(
    xs: np.ndarray,
    zs: np.ndarray,
    velocity_model: np.ndarray,
    travel_times: np.ndarray,
    receiver_xs: np.ndarray,
    receiver_times: np.ndarray,
    ray_paths,
    config: ModelConfig,
    layers=LAYERS
):
    """
    Строит главный итоговый рисунок:
    сверху — кривая времени,
    снизу — слоистая модель и лучи.
    """
    fig = plt.figure(figsize=(10, 8))

    # -----------------------------
    # Верхняя часть: кривая времени.
    # -----------------------------
    ax1 = fig.add_axes([0.10, 0.62, 0.82, 0.30])

    ax1.scatter(receiver_xs, receiver_times, facecolors="none", edgecolors="red")
    ax1.plot(receiver_xs, receiver_times, alpha=0.7)

    ax1.set_title("Travel-time curve")
    ax1.set_xlabel("Receiver x, m")
    ax1.set_ylabel("Time, s")
    ax1.grid(True, alpha=0.3)

    # ------------------------------------------------
    # Нижняя часть: слоистая модель + восстановленные лучи.
    # ------------------------------------------------
    ax2 = fig.add_axes([0.10, 0.08, 0.82, 0.42])

    # Медленность = 1 / скорость.
    slowness = 1.0 / velocity_model

    image = ax2.imshow(
        slowness,
        extent=[xs[0], xs[-1], zs[0], zs[-1]],
        origin="lower",
        aspect="auto",
    )

    colorbar = fig.colorbar(image, ax=ax2, fraction=0.046, pad=0.04)
    colorbar.set_label("Slowness, s/m")

    # Границы слоев.
    for _, z_bottom, _ in layers[:-1]:
        ax2.axhline(z_bottom, color="black", linewidth=0.8, alpha=0.5)

    # Лучи.
    for path in ray_paths:
        path_xs = [xs[ix] for iz, ix in path]
        path_zs = [zs[iz] for iz, ix in path]

        ax2.plot(path_xs, path_zs, color="red", linewidth=0.8, alpha=0.75)

    # Источник.
    ax2.scatter(
        [config.source_x],
        [config.source_z],
        marker="*",
        s=180,
        c="white",
        edgecolors="black",
        zorder=5,
        label="Source",
    )

    # Приемники.
    ax2.scatter(
        receiver_xs,
        np.full_like(receiver_xs, config.receiver_z),
        marker="v",
        s=35,
        c="yellow",
        edgecolors="black",
        zorder=5,
        label="Receivers",
    )

    ax2.set_title("Simple layered model with minimum-time rays")
    ax2.set_xlabel("Distance, m")
    ax2.set_ylabel("Depth, m")
    ax2.invert_yaxis()
    ax2.legend(loc="lower right")

    fig.savefig("layered_model_rays_and_times.png", dpi=200, bbox_inches="tight")
    plt.show()

    # -----------------------------
    # Дополнительная карта времен.
    # -----------------------------
    plt.figure(figsize=(9, 4.8))

    plt.imshow(
        travel_times,
        extent=[xs[0], xs[-1], zs[0], zs[-1]],
        origin="lower",
        aspect="auto",
    )

    plt.colorbar(label="Travel time, s")

    for _, z_bottom, _ in layers[:-1]:
        plt.axhline(z_bottom, color="white", linewidth=0.8, alpha=0.5)

    plt.scatter(
        [config.source_x],
        [config.source_z],
        marker="*",
        s=150,
        c="white",
        edgecolors="black",
        label="Source",
    )

    plt.scatter(
        receiver_xs,
        np.full_like(receiver_xs, config.receiver_z),
        marker="v",
        s=30,
        c="yellow",
        edgecolors="black",
        label="Receivers",
    )

    plt.gca().invert_yaxis()
    plt.title("Travel-time map")
    plt.xlabel("Distance, m")
    plt.ylabel("Depth, m")
    plt.legend(loc="lower right")

    plt.savefig("layered_time_map.png", dpi=200, bbox_inches="tight")
    plt.show()


# ============================================================
# 6. ЗАПУСК ПРОГРАММЫ
# ============================================================

def main():
    config = ModelConfig()

    validate_layers(LAYERS)

    xs, zs, dx, dz = build_grid(config)

    velocity_model = build_velocity_model(xs, zs, LAYERS)

    source_ix = nearest_index(xs, config.source_x)
    source_iz = nearest_index(zs, config.source_z)

    travel_times, parent_iz, parent_ix = dijkstra_with_paths(
        velocity_model,
        dx,
        dz,
        source_iz,
        source_ix
    )

    receiver_xs, receiver_times, ray_paths = get_receiver_times_and_paths(
        xs,
        zs,
        travel_times,
        parent_iz,
        parent_ix,
        config
    )

    print("Слоистая модель:")
    for i, (z_top, z_bottom, velocity) in enumerate(LAYERS, start=1):
        print(
            f"Слой {i}: "
            f"глубина {z_top:.1f}–{z_bottom:.1f} м, "
            f"скорость {velocity:.1f} м/с = {velocity / 1000:.2f} км/с"
        )

    print("\nВремена прихода волны на приемники:")
    print("№\tx, м\tt, с")

    for i, (receiver_x, receiver_time) in enumerate(zip(receiver_xs, receiver_times), start=1):
        print(f"{i:02d}\t{receiver_x:8.1f}\t{receiver_time:.6f}")

    plot_results(
        xs,
        zs,
        velocity_model,
        travel_times,
        receiver_xs,
        receiver_times,
        ray_paths,
        config,
        LAYERS
    )

    print("\nГотово. Сохранены файлы:")
    print("layered_model_rays_and_times.png")
    print("layered_time_map.png")


main()

## Этап 1 — проверка прямой модели

Этап 1 проекта:
проверка прямой модели для разных глубин источника.

Запуск в Google Colab:
просто запустить ячейку.

Результат:
1. comparison_travel_time_curves.png — сравнение travel-time curves.
2. lower_source_rays.png — лучи для источника в нижнем слое.

In [ ]:
import heapq
import math
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np


@dataclass
class ModelConfig:
    x_min: float = 0.0
    x_max: float = 1200.0
    z_min: float = 0.0
    z_max: float = 500.0
    nx: int = 241
    nz: int = 101
    receiver_count: int = 25
    receiver_z: float = 0.0


LAYERS = [
    (0.0, 120.0, 1500.0),
    (120.0, 300.0, 2200.0),
    (300.0, 500.0, 3200.0),
]


def build_grid(config):
    xs = np.linspace(config.x_min, config.x_max, config.nx)
    zs = np.linspace(config.z_min, config.z_max, config.nz)
    dx = xs[1] - xs[0]
    dz = zs[1] - zs[0]
    return xs, zs, dx, dz


def nearest_index(array, value):
    return int(np.argmin(np.abs(array - value)))


def validate_layers(layers):
    previous_velocity = -np.inf
    for i, (z_top, z_bottom, velocity) in enumerate(layers, start=1):
        if z_bottom <= z_top:
            raise ValueError(f"Ошибка в слое {i}: нижняя граница должна быть больше верхней.")
        if not (1000.0 <= velocity <= 10000.0):
            raise ValueError(f"Ошибка в слое {i}: скорость вне диапазона 1000–10000 м/с.")
        if velocity < previous_velocity:
            raise ValueError(f"Ошибка в слое {i}: скорость в нижнем слое меньше, чем выше.")
        previous_velocity = velocity


def velocity_by_depth(z, layers=LAYERS):
    for z_top, z_bottom, velocity in layers:
        if z_top <= z <= z_bottom:
            return velocity
    return layers[-1][2]


def build_velocity_model(xs, zs, layers=LAYERS):
    velocity_model = np.zeros((len(zs), len(xs)))
    for iz, z in enumerate(zs):
        velocity_model[iz, :] = velocity_by_depth(z, layers)
    return velocity_model


def dijkstra_with_paths(velocity_model, dx, dz, source_iz, source_ix):
    nz, nx = velocity_model.shape
    travel_times = np.full((nz, nx), np.inf)
    parent_iz = np.full((nz, nx), -1, dtype=int)
    parent_ix = np.full((nz, nx), -1, dtype=int)

    neighbors = [
        (-1, 0), (1, 0), (0, -1), (0, 1),
        (-1, -1), (-1, 1), (1, -1), (1, 1),
    ]

    travel_times[source_iz, source_ix] = 0.0
    heap = [(0.0, source_iz, source_ix)]

    while heap:
        current_time, iz, ix = heapq.heappop(heap)
        if current_time > travel_times[iz, ix]:
            continue

        for dzi, dxi in neighbors:
            next_iz = iz + dzi
            next_ix = ix + dxi
            if not (0 <= next_iz < nz and 0 <= next_ix < nx):
                continue

            step_distance = math.sqrt((dxi * dx) ** 2 + (dzi * dz) ** 2)
            average_velocity = 0.5 * (velocity_model[iz, ix] + velocity_model[next_iz, next_ix])
            step_time = step_distance / average_velocity
            new_time = current_time + step_time

            if new_time < travel_times[next_iz, next_ix]:
                travel_times[next_iz, next_ix] = new_time
                parent_iz[next_iz, next_ix] = iz
                parent_ix[next_iz, next_ix] = ix
                heapq.heappush(heap, (new_time, next_iz, next_ix))

    return travel_times, parent_iz, parent_ix


def reconstruct_path(target_iz, target_ix, parent_iz, parent_ix):
    path = []
    iz, ix = int(target_iz), int(target_ix)
    while iz != -1 and ix != -1:
        path.append((iz, ix))
        previous_iz = parent_iz[iz, ix]
        previous_ix = parent_ix[iz, ix]
        if previous_iz == -1 and previous_ix == -1:
            break
        iz, ix = previous_iz, previous_ix
    path.reverse()
    return path


def get_receiver_coordinates(xs, config):
    return np.linspace(config.x_min + 50.0, config.x_max - 50.0, config.receiver_count)


def compute_for_source(source_x, source_z, config, layers=LAYERS):
    xs, zs, dx, dz = build_grid(config)
    velocity_model = build_velocity_model(xs, zs, layers)

    source_ix = nearest_index(xs, source_x)
    source_iz = nearest_index(zs, source_z)

    travel_times, parent_iz, parent_ix = dijkstra_with_paths(
        velocity_model, dx, dz, source_iz, source_ix
    )

    receiver_xs = get_receiver_coordinates(xs, config)
    receiver_iz = nearest_index(zs, config.receiver_z)

    receiver_times = []
    ray_paths = []

    for receiver_x in receiver_xs:
        receiver_ix = nearest_index(xs, receiver_x)
        receiver_times.append(travel_times[receiver_iz, receiver_ix])
        ray_paths.append(reconstruct_path(receiver_iz, receiver_ix, parent_iz, parent_ix))

    return {
        "xs": xs,
        "zs": zs,
        "velocity_model": velocity_model,
        "receiver_xs": receiver_xs,
        "receiver_times": np.array(receiver_times),
        "ray_paths": ray_paths,
        "source_x": source_x,
        "source_z": source_z,
    }


def plot_comparison(result_middle, result_lower):
    plt.figure(figsize=(10, 5))
    plt.plot(
        result_middle["receiver_xs"],
        result_middle["receiver_times"],
        marker="o",
        label=f"Источник z={result_middle['source_z']} м"
    )
    plt.plot(
        result_lower["receiver_xs"],
        result_lower["receiver_times"],
        marker="o",
        label=f"Источник z={result_lower['source_z']} м"
    )
    plt.title("Сравнение travel-time curve для разных глубин источника")
    plt.xlabel("Координата приемника x, м")
    plt.ylabel("Время прихода, с")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.savefig("comparison_travel_time_curves.png", dpi=200, bbox_inches="tight")
    plt.show()


def plot_lower_source_rays(result, layers=LAYERS):
    xs = result["xs"]
    zs = result["zs"]
    velocity_model = result["velocity_model"]
    receiver_xs = result["receiver_xs"]
    ray_paths = result["ray_paths"]
    source_x = result["source_x"]
    source_z = result["source_z"]

    slowness = 1.0 / velocity_model

    plt.figure(figsize=(10, 5.5))
    plt.imshow(
        slowness,
        extent=[xs[0], xs[-1], zs[0], zs[-1]],
        origin="lower",
        aspect="auto"
    )
    plt.colorbar(label="Медленность, с/м")

    for _, z_bottom, _ in layers[:-1]:
        plt.axhline(z_bottom, color="black", linewidth=0.8, alpha=0.6)

    for path in ray_paths:
        path_xs = [xs[ix] for iz, ix in path]
        path_zs = [zs[iz] for iz, ix in path]
        plt.plot(path_xs, path_zs, color="red", linewidth=0.8, alpha=0.75)

    plt.scatter([source_x], [source_z], marker="*", s=180, c="white", edgecolors="black", zorder=5, label="Источник")
    plt.scatter(receiver_xs, np.zeros_like(receiver_xs), marker="v", s=35, c="yellow", edgecolors="black", zorder=5, label="Приемники")

    plt.gca().invert_yaxis()
    plt.title("Источник в нижнем слое: слоистая модель и лучи")
    plt.xlabel("x, м")
    plt.ylabel("z, м, глубина")
    plt.legend(loc="lower right")
    plt.savefig("lower_source_rays.png", dpi=200, bbox_inches="tight")
    plt.show()


def print_travel_times(result, title):
    print(title)
    print("№\tx, м\tt, с")
    for i, (x, t) in enumerate(zip(result["receiver_xs"], result["receiver_times"]), start=1):
        print(f"{i:02d}\t{x:8.1f}\t{t:.6f}")
    print()


def main():
    validate_layers(LAYERS)
    config = ModelConfig()

    # Старый вариант: источник в среднем слое.
    result_middle = compute_for_source(source_x=600.0, source_z=150.0, config=config)

    # Новый вариант по заданию Евгения: источник в самом нижнем слое.
    result_lower = compute_for_source(source_x=600.0, source_z=400.0, config=config)

    print("Слоистая модель:")
    for i, (z_top, z_bottom, velocity) in enumerate(LAYERS, start=1):
        print(f"Слой {i}: {z_top:.0f}–{z_bottom:.0f} м, v={velocity:.0f} м/с")
    print()

    print_travel_times(result_middle, "Источник в среднем слое:")
    print_travel_times(result_lower, "Источник в нижнем слое:")

    plot_comparison(result_middle, result_lower)
    plot_lower_source_rays(result_lower)

    print("Готово. Сохранены файлы:")
    print("comparison_travel_time_curves.png")
    print("lower_source_rays.png")


main()

## Этап 2.1 — расширенная классификация

Этап 2.1:
расширенная проверка классификатора.

Что изменено по сравнению с прошлой версией:
1. N_PER_CLASS увеличен до 1000.
2. Основной датасет теперь содержит 3000 примеров:
   - 1000 однородных;
   - 1000 горизонтально-слоистых;
   - 1000 контрастных двухслойных.
3. Добавлена проверка устойчивости на нескольких random seed.
4. Добавлена итоговая таблица по всем запускам.
5. Сохраняются CSV-файлы с результатами.

Запуск в Google Colab:
просто запустить ячейку.

Если будет долго считаться:
можно временно уменьшить SEED_CHECK_N_PER_CLASS до 300 или 500.

In [ ]:
import heapq
import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

@dataclass
class ModelConfig:
    x_min: float = 0.0
    x_max: float = 1200.0
    z_min: float = 0.0
    z_max: float = 500.0

    # Грубая сетка для быстрой генерации датасета.
    # Для красивых лучей используем более плотную сетку,
    # для классификатора важна скорость расчета многих примеров.
    nx: int = 61
    nz: int = 51

    source_x: float = 600.0
    source_z: float = 400.0

    receiver_count: int = 25
    receiver_z: float = 0.0


CONFIG = ModelConfig()

# Основная проверка: 1000 на каждый класс = 3000 всего.
N_PER_CLASS = 1000

# Проверка устойчивости на других seed.
# Если будет долго считаться, можно поставить 500.
SEED_CHECK_N_PER_CLASS = 1000

MAIN_SEED = 42
EXTRA_SEEDS = [7, 123]

CLASS_NAMES = {
    0: "однородный",
    1: "горизонтально-слоистый",
    2: "контрастный двухслойный",
}


# ============================================================
# 2. СЕТКА И ПРЯМАЯ МОДЕЛЬ
# ============================================================

def build_grid(config):
    xs = np.linspace(config.x_min, config.x_max, config.nx)
    zs = np.linspace(config.z_min, config.z_max, config.nz)
    dx = xs[1] - xs[0]
    dz = zs[1] - zs[0]
    return xs, zs, dx, dz


XS, ZS, DX, DZ = build_grid(CONFIG)

SOURCE_IX = int(np.argmin(np.abs(XS - CONFIG.source_x)))
SOURCE_IZ = int(np.argmin(np.abs(ZS - CONFIG.source_z)))

RECEIVER_XS = np.linspace(
    CONFIG.x_min + 50.0,
    CONFIG.x_max - 50.0,
    CONFIG.receiver_count
)

RECEIVER_IZ = int(np.argmin(np.abs(ZS - CONFIG.receiver_z)))
RECEIVER_IXS = [int(np.argmin(np.abs(XS - x))) for x in RECEIVER_XS]


def velocity_by_depth(z, layers):
    for z_top, z_bottom, velocity in layers:
        if z_top <= z <= z_bottom:
            return velocity
    return layers[-1][2]


def build_velocity_model(layers):
    velocity_model = np.zeros((len(ZS), len(XS)))

    for iz, z in enumerate(ZS):
        velocity_model[iz, :] = velocity_by_depth(z, layers)

    return velocity_model


def dijkstra_travel_times(velocity_model):
    nz, nx = velocity_model.shape

    travel_times = np.full((nz, nx), np.inf)
    travel_times[SOURCE_IZ, SOURCE_IX] = 0.0

    neighbors = [
        (-1, 0), (1, 0), (0, -1), (0, 1),
        (-1, -1), (-1, 1), (1, -1), (1, 1),
    ]

    heap = [(0.0, SOURCE_IZ, SOURCE_IX)]

    while heap:
        current_time, iz, ix = heapq.heappop(heap)

        if current_time > travel_times[iz, ix]:
            continue

        for dzi, dxi in neighbors:
            next_iz = iz + dzi
            next_ix = ix + dxi

            if not (0 <= next_iz < nz and 0 <= next_ix < nx):
                continue

            step_distance = math.sqrt((dxi * DX) ** 2 + (dzi * DZ) ** 2)

            average_velocity = 0.5 * (
                velocity_model[iz, ix] + velocity_model[next_iz, next_ix]
            )

            step_time = step_distance / average_velocity
            new_time = current_time + step_time

            if new_time < travel_times[next_iz, next_ix]:
                travel_times[next_iz, next_ix] = new_time
                heapq.heappush(heap, (new_time, next_iz, next_ix))

    receiver_times = np.array([
        travel_times[RECEIVER_IZ, receiver_ix]
        for receiver_ix in RECEIVER_IXS
    ])

    return receiver_times


def compute_travel_time_curve(layers):
    velocity_model = build_velocity_model(layers)
    return dijkstra_travel_times(velocity_model)


# ============================================================
# 3. ГЕНЕРАЦИЯ ПРОФИЛЕЙ
# ============================================================

def generate_homogeneous_profile():
    """
    Класс 0: однородный профиль.
    Одна скорость на всю глубину.
    """
    velocity = random.uniform(1600.0, 3800.0)

    layers = [
        (0.0, CONFIG.z_max, velocity)
    ]

    return layers


def generate_layered_profile():
    """
    Класс 1: горизонтально-слоистый профиль.
    3–5 слоев, скорость постепенно увеличивается с глубиной.
    """
    layer_count = random.randint(3, 5)

    possible_boundaries = list(np.arange(80.0, 440.0, 20.0))
    inner_boundaries = sorted(random.sample(possible_boundaries, layer_count - 1))

    boundaries = [0.0] + inner_boundaries + [CONFIG.z_max]

    first_velocity = random.uniform(1300.0, 2300.0)

    velocities = [first_velocity]

    for _ in range(layer_count - 1):
        increment = random.uniform(200.0, 700.0)
        velocities.append(min(10000.0, velocities[-1] + increment))

    layers = []

    for i in range(layer_count):
        layers.append(
            (boundaries[i], boundaries[i + 1], velocities[i])
        )

    return layers


def generate_contrast_two_layer_profile():
    """
    Класс 2: контрастный двухслойный профиль.
    Два слоя с резким скачком скорости.
    """
    boundary = random.uniform(160.0, 340.0)

    upper_velocity = random.uniform(1300.0, 2200.0)
    lower_velocity = random.uniform(3800.0, 6500.0)

    layers = [
        (0.0, boundary, upper_velocity),
        (boundary, CONFIG.z_max, lower_velocity),
    ]

    return layers


def generate_profile_by_class(class_label):
    if class_label == 0:
        return generate_homogeneous_profile()
    if class_label == 1:
        return generate_layered_profile()
    if class_label == 2:
        return generate_contrast_two_layer_profile()

    raise ValueError("Неизвестный класс профиля.")


# ============================================================
# 4. ГЕНЕРАЦИЯ ДАТАСЕТА
# ============================================================

def generate_dataset(n_per_class, seed, verbose=True):
    random.seed(seed)
    np.random.seed(seed)

    X = []
    y = []
    all_layers = []

    total = n_per_class * 3
    counter = 0

    for class_label in [0, 1, 2]:
        for _ in range(n_per_class):
            layers = generate_profile_by_class(class_label)
            times = compute_travel_time_curve(layers)

            X.append(times)
            y.append(class_label)
            all_layers.append(layers)

            counter += 1

            if verbose and counter % 300 == 0:
                print(f"Seed {seed}: сгенерировано {counter}/{total} примеров...")

    return np.array(X), np.array(y), all_layers


def save_dataset_to_csv(X, y, filename):
    columns = [f"t{i}" for i in range(1, X.shape[1] + 1)]

    df = pd.DataFrame(X, columns=columns)
    df["class"] = y

    df.to_csv(filename, index=False)

    print(f"Датасет сохранен в файл: {filename}")

    return df


# ============================================================
# 5. ПРОВЕРКИ ДАТАСЕТА
# ============================================================

def check_dataset(X, y):
    print("Размер X:", X.shape)
    print("Размер y:", y.shape)

    print("Распределение классов:")
    for class_label in [0, 1, 2]:
        print(class_label, CLASS_NAMES[class_label], ":", np.sum(y == class_label))

    print("Проверка NaN:", np.isnan(X).any())
    print("Проверка inf:", np.isinf(X).any())

    print("Минимальное время:", X.min())
    print("Максимальное время:", X.max())

    if np.isnan(X).any() or np.isinf(X).any():
        raise ValueError("В датасете есть NaN или inf.")

    if X.min() <= 0:
        raise ValueError("В датасете есть неположительные времена.")


# ============================================================
# 6. ПРИЗНАКИ
# ============================================================

def build_features(X, feature_mode):
    features = [X]

    if feature_mode >= 1:
        first_derivative = np.gradient(X, axis=1)
        features.append(first_derivative)

    if feature_mode >= 2:
        first_derivative = np.gradient(X, axis=1)
        second_derivative = np.gradient(first_derivative, axis=1)
        features.append(second_derivative)

    return np.hstack(features)


def feature_mode_name(feature_mode):
    if feature_mode == 0:
        return "times"
    if feature_mode == 1:
        return "times + первая производная"
    if feature_mode == 2:
        return "times + первая + вторая производная"
    return "unknown"


# ============================================================
# 7. КЛАССИФИКАТОРЫ
# ============================================================

def train_and_evaluate_classifier(X, y, classifier, classifier_name, feature_mode, seed, detailed=False):
    features = build_features(X, feature_mode)

    X_train, X_test, y_train, y_test = train_test_split(
        features,
        y,
        test_size=0.25,
        random_state=seed,
        stratify=y
    )

    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    matrix = confusion_matrix(y_test, y_pred)

    if detailed:
        print("=" * 80)
        print(f"Seed: {seed}")
        print(f"Классификатор: {classifier_name}")
        print(f"Признаки: {feature_mode_name(feature_mode)}")
        print(f"Accuracy: {accuracy:.3f}")
        print()
        print("Confusion matrix:")
        print(matrix)
        print()
        print("Classification report:")
        print(classification_report(
            y_test,
            y_pred,
            target_names=[CLASS_NAMES[0], CLASS_NAMES[1], CLASS_NAMES[2]]
        ))

    return {
        "seed": seed,
        "classifier": classifier_name,
        "features": feature_mode_name(feature_mode),
        "accuracy": accuracy,
        "confusion_matrix": matrix,
    }


def run_experiments(X, y, seed, detailed=False):
    results = []

    for feature_mode in [0, 1, 2]:
        results.append(
            train_and_evaluate_classifier(
                X,
                y,
                GaussianNB(),
                "Gaussian Naive Bayes",
                feature_mode,
                seed,
                detailed=detailed
            )
        )

        results.append(
            train_and_evaluate_classifier(
                X,
                y,
                DecisionTreeClassifier(max_depth=6, random_state=seed),
                "Decision Tree",
                feature_mode,
                seed,
                detailed=detailed
            )
        )

    return results


def results_to_dataframe(results):
    return pd.DataFrame([
        {
            "seed": item["seed"],
            "classifier": item["classifier"],
            "features": item["features"],
            "accuracy": item["accuracy"],
        }
        for item in results
    ])


# ============================================================
# 8. ВИЗУАЛИЗАЦИЯ ПРИМЕРОВ
# ============================================================

def plot_velocity_profile(layers, title):
    depths = []
    velocities = []

    for z_top, z_bottom, velocity in layers:
        depths.extend([z_top, z_bottom])
        velocities.extend([velocity, velocity])

    plt.plot(velocities, depths, marker="o")
    plt.gca().invert_yaxis()
    plt.xlabel("Скорость, м/с")
    plt.ylabel("Глубина, м")
    plt.title(title)
    plt.grid(True, alpha=0.3)


def plot_example_profiles(seed=MAIN_SEED):
    random.seed(seed)
    np.random.seed(seed)

    plt.figure(figsize=(12, 10))

    for row, class_label in enumerate([0, 1, 2]):
        layers = generate_profile_by_class(class_label)
        times = compute_travel_time_curve(layers)

        plt.subplot(3, 2, row * 2 + 1)
        plot_velocity_profile(layers, f"Класс {class_label}: {CLASS_NAMES[class_label]}")

        plt.subplot(3, 2, row * 2 + 2)
        plt.plot(RECEIVER_XS, times, marker="o")
        plt.xlabel("x приемника, м")
        plt.ylabel("Время, с")
        plt.title("Travel-time curve")
        plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("example_profiles_and_curves_1000.png", dpi=200, bbox_inches="tight")
    plt.show()

    print("Примеры профилей сохранены в файл: example_profiles_and_curves_1000.png")


# ============================================================
# 9. ЗАПУСК ОСНОВНОЙ ПРОВЕРКИ
# ============================================================

print("Рисуем примеры трех классов...")
plot_example_profiles(seed=MAIN_SEED)

print()
print("=" * 80)
print("ОСНОВНОЙ ЗАПУСК")
print(f"N_PER_CLASS = {N_PER_CLASS}, seed = {MAIN_SEED}")
print("=" * 80)

X_main, y_main, layers_main = generate_dataset(
    n_per_class=N_PER_CLASS,
    seed=MAIN_SEED,
    verbose=True
)

print()
check_dataset(X_main, y_main)

df_main = save_dataset_to_csv(
    X_main,
    y_main,
    filename="travel_time_dataset_3000_seed42.csv"
)

print()
print("Первые строки датасета:")
display(df_main.head())

print()
print("Обучаем классификаторы на основном датасете...")
main_results = run_experiments(
    X_main,
    y_main,
    seed=MAIN_SEED,
    detailed=True
)

main_results_df = results_to_dataframe(main_results)

print()
print("Итоговая таблица основного запуска:")
display(main_results_df)

main_results_df.to_csv("classifier_results_3000_seed42.csv", index=False)


# ============================================================
# 10. ПРОВЕРКА УСТОЙЧИВОСТИ НА ДРУГИХ SEED
# ============================================================

print()
print("=" * 80)
print("ПРОВЕРКА УСТОЙЧИВОСТИ НА ДРУГИХ RANDOM SEED")
print("=" * 80)

all_seed_results = []

# Добавляем основной запуск в общую таблицу.
all_seed_results.extend(main_results)

for seed in EXTRA_SEEDS:
    print()
    print("-" * 80)
    print(f"Проверка seed = {seed}, N_PER_CLASS = {SEED_CHECK_N_PER_CLASS}")
    print("-" * 80)

    X_seed, y_seed, _ = generate_dataset(
        n_per_class=SEED_CHECK_N_PER_CLASS,
        seed=seed,
        verbose=True
    )

    check_dataset(X_seed, y_seed)

    seed_results = run_experiments(
        X_seed,
        y_seed,
        seed=seed,
        detailed=False
    )

    seed_results_df = results_to_dataframe(seed_results)
    display(seed_results_df)

    all_seed_results.extend(seed_results)

all_seed_results_df = results_to_dataframe(all_seed_results)

print()
print("Общая таблица по всем seed:")
display(all_seed_results_df)

all_seed_results_df.to_csv("classifier_results_all_seeds.csv", index=False)

print()
print("Средняя accuracy по seed:")
summary_df = (
    all_seed_results_df
    .groupby(["classifier", "features"], as_index=False)
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        min_accuracy=("accuracy", "min"),
        max_accuracy=("accuracy", "max"),
    )
    .sort_values("mean_accuracy", ascending=False)
)

display(summary_df)
summary_df.to_csv("classifier_results_summary_all_seeds.csv", index=False)


print()
print("Готово. Основные файлы:")
print("travel_time_dataset_3000_seed42.csv")
print("classifier_results_3000_seed42.csv")
print("classifier_results_all_seeds.csv")
print("classifier_results_summary_all_seeds.csv")
print("example_profiles_and_curves_1000.png")

## Этап 7.3 — устойчивость классификатора к шуму

Этап 7.3:
расширенная и более аккуратная проверка устойчивости классификаторов к шуму.

Зачем нужен этот этап:
на этапе 7.1 стало видно, что:
1) на чистых данных производные travel-time curve сильно помогают;
2) при шуме производные могут становиться нестабильными;
3) корректная фильтрация train/test помогает;
4) noise augmentation с шумами 1%, 3%, 5% не покрывает 10% шум.

Что улучшено в 7.3:
1. Нет утечки данных:
   сначала делим clean dataset на train/test, и только потом создаем noisy/augmented train.
   Test set никогда не попадает в augmentation.
2. Шум в train и test независимый.
3. Фильтрация применяется одинаково к train и test.
4. Проверяем больше стратегий обучения:
   - clean train;
   - matched noisy train;
   - augmentation 1/3/5;
   - augmentation 1/3/5/10;
   - random augmentation 0-10%;
   - эти же варианты с фильтрацией.
5. Проверяем разные признаки:
   - times;
   - times + d1;
   - times + d1 + d2.
6. Проверяем несколько random seed:
   - 42, 7, 123.
7. Добавлен Random Forest как дополнительный контрольный классификатор.
   Основные классификаторы для отчета остаются Gaussian Naive Bayes и Decision Tree,
   Random Forest можно использовать как дополнительный эксперимент.
8. В конце строятся сводные таблицы:
   - лучшая конфигурация для каждого уровня шума;
   - лучшие конфигурации по средней accuracy на шумных данных;
   - лучшие конфигурации по worst-case accuracy.
9. Сохраняются CSV и графики.

Если будет долго выполняться:
- поставить N_PER_CLASS = 500;
- или USE_RANDOM_FOREST = False.

In [ ]:
import heapq
import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

@dataclass
class ModelConfig:
    x_min: float = 0.0
    x_max: float = 1200.0
    z_min: float = 0.0
    z_max: float = 500.0

    # Грубая сетка для генерации большого числа моделей.
    nx: int = 61
    nz: int = 51

    source_x: float = 600.0
    source_z: float = 400.0

    receiver_count: int = 25
    receiver_z: float = 0.0


CONFIG = ModelConfig()

N_PER_CLASS = 1000
GENERATION_SEED = 42

EVAL_SEEDS = [42, 7, 123]

NOISE_LEVELS = [0.0, 0.01, 0.03, 0.05, 0.10]

USE_RANDOM_FOREST = True

CLASS_NAMES = {
    0: "однородный",
    1: "горизонтально-слоистый",
    2: "контрастный двухслойный",
}

FEATURE_MODES = {
    0: "times",
    1: "times + d1",
    2: "times + d1 + d2",
}


# ============================================================
# 2. СЕТКА И ПРЯМАЯ МОДЕЛЬ
# ============================================================

def build_grid(config):
    xs = np.linspace(config.x_min, config.x_max, config.nx)
    zs = np.linspace(config.z_min, config.z_max, config.nz)
    dx = xs[1] - xs[0]
    dz = zs[1] - zs[0]
    return xs, zs, dx, dz


XS, ZS, DX, DZ = build_grid(CONFIG)

SOURCE_IX = int(np.argmin(np.abs(XS - CONFIG.source_x)))
SOURCE_IZ = int(np.argmin(np.abs(ZS - CONFIG.source_z)))

RECEIVER_XS = np.linspace(
    CONFIG.x_min + 50.0,
    CONFIG.x_max - 50.0,
    CONFIG.receiver_count
)

RECEIVER_IZ = int(np.argmin(np.abs(ZS - CONFIG.receiver_z)))
RECEIVER_IXS = [int(np.argmin(np.abs(XS - x))) for x in RECEIVER_XS]


def velocity_by_depth(z, layers):
    for z_top, z_bottom, velocity in layers:
        if z_top <= z <= z_bottom:
            return velocity
    return layers[-1][2]


def build_velocity_model(layers):
    velocity_model = np.zeros((len(ZS), len(XS)))

    for iz, z in enumerate(ZS):
        velocity_model[iz, :] = velocity_by_depth(z, layers)

    return velocity_model


def dijkstra_travel_times(velocity_model):
    nz, nx = velocity_model.shape

    travel_times = np.full((nz, nx), np.inf)
    travel_times[SOURCE_IZ, SOURCE_IX] = 0.0

    neighbors = [
        (-1, 0), (1, 0), (0, -1), (0, 1),
        (-1, -1), (-1, 1), (1, -1), (1, 1),
    ]

    heap = [(0.0, SOURCE_IZ, SOURCE_IX)]

    while heap:
        current_time, iz, ix = heapq.heappop(heap)

        if current_time > travel_times[iz, ix]:
            continue

        for dzi, dxi in neighbors:
            next_iz = iz + dzi
            next_ix = ix + dxi

            if not (0 <= next_iz < nz and 0 <= next_ix < nx):
                continue

            step_distance = math.sqrt((dxi * DX) ** 2 + (dzi * DZ) ** 2)

            average_velocity = 0.5 * (
                velocity_model[iz, ix] + velocity_model[next_iz, next_ix]
            )

            step_time = step_distance / average_velocity
            new_time = current_time + step_time

            if new_time < travel_times[next_iz, next_ix]:
                travel_times[next_iz, next_ix] = new_time
                heapq.heappush(heap, (new_time, next_iz, next_ix))

    receiver_times = np.array([
        travel_times[RECEIVER_IZ, receiver_ix]
        for receiver_ix in RECEIVER_IXS
    ])

    return receiver_times


def compute_travel_time_curve(layers):
    velocity_model = build_velocity_model(layers)
    return dijkstra_travel_times(velocity_model)


# ============================================================
# 3. ГЕНЕРАЦИЯ ПРОФИЛЕЙ
# ============================================================

def generate_homogeneous_profile():
    velocity = random.uniform(1600.0, 3800.0)
    return [(0.0, CONFIG.z_max, velocity)]


def generate_layered_profile():
    layer_count = random.randint(3, 5)

    possible_boundaries = list(np.arange(80.0, 440.0, 20.0))
    inner_boundaries = sorted(random.sample(possible_boundaries, layer_count - 1))

    boundaries = [0.0] + inner_boundaries + [CONFIG.z_max]

    first_velocity = random.uniform(1300.0, 2300.0)
    velocities = [first_velocity]

    for _ in range(layer_count - 1):
        increment = random.uniform(200.0, 700.0)
        velocities.append(min(10000.0, velocities[-1] + increment))

    layers = []

    for i in range(layer_count):
        layers.append((boundaries[i], boundaries[i + 1], velocities[i]))

    return layers


def generate_contrast_two_layer_profile():
    boundary = random.uniform(160.0, 340.0)

    upper_velocity = random.uniform(1300.0, 2200.0)
    lower_velocity = random.uniform(3800.0, 6500.0)

    return [
        (0.0, boundary, upper_velocity),
        (boundary, CONFIG.z_max, lower_velocity),
    ]


def generate_profile_by_class(class_label):
    if class_label == 0:
        return generate_homogeneous_profile()
    if class_label == 1:
        return generate_layered_profile()
    if class_label == 2:
        return generate_contrast_two_layer_profile()

    raise ValueError("Неизвестный класс профиля.")


def generate_dataset(n_per_class, seed):
    random.seed(seed)
    np.random.seed(seed)

    X = []
    y = []

    total = n_per_class * 3
    counter = 0

    for class_label in [0, 1, 2]:
        for _ in range(n_per_class):
            layers = generate_profile_by_class(class_label)
            times = compute_travel_time_curve(layers)

            X.append(times)
            y.append(class_label)

            counter += 1

            if counter % 300 == 0:
                print(f"Сгенерировано {counter}/{total} примеров...")

    return np.array(X), np.array(y)


def check_dataset(X, y):
    print("Размер X:", X.shape)
    print("Размер y:", y.shape)

    print("Распределение классов:")
    for class_label in [0, 1, 2]:
        print(class_label, CLASS_NAMES[class_label], ":", np.sum(y == class_label))

    print("Проверка NaN:", np.isnan(X).any())
    print("Проверка inf:", np.isinf(X).any())
    print("Минимальное время:", X.min())
    print("Максимальное время:", X.max())

    if np.isnan(X).any() or np.isinf(X).any():
        raise ValueError("В датасете есть NaN или inf.")

    if X.min() <= 0:
        raise ValueError("В датасете есть неположительные времена.")


# ============================================================
# 4. ШУМ, ФИЛЬТРАЦИЯ, AUGMENTATION
# ============================================================

def add_relative_gaussian_noise(X, noise_level, seed):
    """
    Относительный гауссов шум:
        X_noisy = X + Normal(0, noise_level * X)
    """
    if noise_level == 0:
        return X.copy()

    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=0.0, scale=noise_level * X, size=X.shape)
    X_noisy = X + noise

    return np.maximum(X_noisy, 1e-6)


def add_random_relative_gaussian_noise(X, max_noise_level, seed):
    """
    Для каждой travel-time curve случайно выбирается уровень шума
    от 0 до max_noise_level, затем добавляется гауссов шум.

    Это более разнообразная augmentation-стратегия.
    """
    if max_noise_level == 0:
        return X.copy()

    rng = np.random.default_rng(seed)

    # Для каждой строки свой уровень шума.
    row_noise_levels = rng.uniform(
        low=0.0,
        high=max_noise_level,
        size=(X.shape[0], 1)
    )

    noise = rng.normal(loc=0.0, scale=row_noise_levels * X, size=X.shape)
    X_noisy = X + noise

    return np.maximum(X_noisy, 1e-6)


def moving_average_filter(X, window_size=3):
    """
    Сглаживание скользящим средним.
    Фильтр надо применять одинаково к train и test.
    """
    if window_size <= 1:
        return X.copy()

    pad = window_size // 2
    X_padded = np.pad(X, ((0, 0), (pad, pad)), mode="edge")

    filtered = np.zeros_like(X)

    for i in range(X.shape[1]):
        filtered[:, i] = np.mean(X_padded[:, i:i + window_size], axis=1)

    return filtered


def make_fixed_noise_augmentation(X_train_clean, y_train, noise_levels, seed_base):
    """
    Делает train augmentation:
        clean + копии с заданными уровнями шума.
    """
    X_parts = [X_train_clean.copy()]
    y_parts = [y_train.copy()]

    for i, noise_level in enumerate(noise_levels):
        X_noisy = add_relative_gaussian_noise(
            X_train_clean,
            noise_level=noise_level,
            seed=seed_base + i * 1000 + int(noise_level * 10000)
        )

        X_parts.append(X_noisy)
        y_parts.append(y_train.copy())

    return np.vstack(X_parts), np.concatenate(y_parts)


def make_random_noise_augmentation(X_train_clean, y_train, max_noise_level, copy_count, seed_base):
    """
    Делает train augmentation:
        clean + copy_count копий, где для каждой curve шум случайный от 0 до max_noise_level.
    """
    X_parts = [X_train_clean.copy()]
    y_parts = [y_train.copy()]

    for i in range(copy_count):
        X_noisy = add_random_relative_gaussian_noise(
            X_train_clean,
            max_noise_level=max_noise_level,
            seed=seed_base + i * 1000 + 777
        )

        X_parts.append(X_noisy)
        y_parts.append(y_train.copy())

    return np.vstack(X_parts), np.concatenate(y_parts)


# ============================================================
# 5. ПРИЗНАКИ И КЛАССИФИКАТОРЫ
# ============================================================

def build_features(X, feature_mode):
    features = [X]

    if feature_mode >= 1:
        d1 = np.gradient(X, axis=1)
        features.append(d1)

    if feature_mode >= 2:
        d1 = np.gradient(X, axis=1)
        d2 = np.gradient(d1, axis=1)
        features.append(d2)

    return np.hstack(features)


def get_classifier_names():
    names = ["Gaussian Naive Bayes", "Decision Tree"]

    if USE_RANDOM_FOREST:
        names.append("Random Forest")

    return names


def create_classifier(classifier_name, seed):
    if classifier_name == "Gaussian Naive Bayes":
        return GaussianNB()

    if classifier_name == "Decision Tree":
        return DecisionTreeClassifier(max_depth=6, random_state=seed)

    if classifier_name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=150,
            max_depth=8,
            random_state=seed,
            n_jobs=-1
        )

    raise ValueError("Неизвестный классификатор.")


def evaluate_model(
    X_train_raw,
    y_train,
    X_test_raw,
    y_test,
    classifier_name,
    feature_mode,
    seed
):
    X_train = build_features(X_train_raw, feature_mode)
    X_test = build_features(X_test_raw, feature_mode)

    classifier = create_classifier(classifier_name, seed)
    classifier.fit(X_train, y_train)

    y_pred = classifier.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    matrix = confusion_matrix(y_test, y_pred)

    return accuracy, matrix, y_pred


# ============================================================
# 6. СТРАТЕГИИ ОБУЧЕНИЯ
# ============================================================

def prepare_training_strategy(
    strategy_name,
    X_train_clean,
    y_train,
    noise_level,
    seed
):
    """
    Возвращает X_train_prepared, y_train_prepared.

    Некоторые стратегии зависят от noise_level:
    например matched noisy train.
    """
    if strategy_name == "clean_train":
        return X_train_clean.copy(), y_train.copy()

    if strategy_name == "matched_noisy_train":
        X_train_noisy = add_relative_gaussian_noise(
            X_train_clean,
            noise_level=noise_level,
            seed=seed + int(noise_level * 10000) + 100
        )
        return X_train_noisy, y_train.copy()

    if strategy_name == "aug_fixed_1_3_5":
        return make_fixed_noise_augmentation(
            X_train_clean,
            y_train,
            noise_levels=[0.01, 0.03, 0.05],
            seed_base=seed + 200
        )

    if strategy_name == "aug_fixed_1_3_5_10":
        return make_fixed_noise_augmentation(
            X_train_clean,
            y_train,
            noise_levels=[0.01, 0.03, 0.05, 0.10],
            seed_base=seed + 300
        )

    if strategy_name == "aug_random_0_10":
        return make_random_noise_augmentation(
            X_train_clean,
            y_train,
            max_noise_level=0.10,
            copy_count=4,
            seed_base=seed + 400
        )

    if strategy_name == "matched_noisy_train_filtered":
        X_train_noisy = add_relative_gaussian_noise(
            X_train_clean,
            noise_level=noise_level,
            seed=seed + int(noise_level * 10000) + 500
        )
        return moving_average_filter(X_train_noisy), y_train.copy()

    if strategy_name == "aug_fixed_1_3_5_10_filtered":
        X_aug, y_aug = make_fixed_noise_augmentation(
            X_train_clean,
            y_train,
            noise_levels=[0.01, 0.03, 0.05, 0.10],
            seed_base=seed + 600
        )
        return moving_average_filter(X_aug), y_aug

    if strategy_name == "aug_random_0_10_filtered":
        X_aug, y_aug = make_random_noise_augmentation(
            X_train_clean,
            y_train,
            max_noise_level=0.10,
            copy_count=4,
            seed_base=seed + 700
        )
        return moving_average_filter(X_aug), y_aug

    raise ValueError(f"Неизвестная стратегия: {strategy_name}")


def prepare_test_for_strategy(strategy_name, X_test_clean, noise_level, seed):
    """
    Возвращает X_test_prepared.

    Если стратегия filtered, то тест тоже фильтруется.
    """
    X_test_noisy = add_relative_gaussian_noise(
        X_test_clean,
        noise_level=noise_level,
        seed=seed + int(noise_level * 10000) + 999
    )

    if strategy_name.endswith("_filtered"):
        return moving_average_filter(X_test_noisy)

    return X_test_noisy


def get_strategy_names():
    return [
        "clean_train",
        "matched_noisy_train",
        "aug_fixed_1_3_5",
        "aug_fixed_1_3_5_10",
        "aug_random_0_10",
        "matched_noisy_train_filtered",
        "aug_fixed_1_3_5_10_filtered",
        "aug_random_0_10_filtered",
    ]


# ============================================================
# 7. ОСНОВНОЙ ЭКСПЕРИМЕНТ
# ============================================================

def run_stage_7_2_experiment(X, y):
    rows = []
    details = {}

    classifier_names = get_classifier_names()
    strategy_names = get_strategy_names()

    for eval_seed in EVAL_SEEDS:
        print()
        print("=" * 80)
        print(f"Evaluation seed = {eval_seed}")
        print("=" * 80)

        X_train_clean, X_test_clean, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.25,
            random_state=eval_seed,
            stratify=y
        )

        for noise_level in NOISE_LEVELS:
            print(f"  Noise level = {int(noise_level * 100)}%")

            for strategy_name in strategy_names:
                X_train_prepared, y_train_prepared = prepare_training_strategy(
                    strategy_name,
                    X_train_clean,
                    y_train,
                    noise_level,
                    seed=eval_seed
                )

                X_test_prepared = prepare_test_for_strategy(
                    strategy_name,
                    X_test_clean,
                    noise_level,
                    seed=eval_seed
                )

                for feature_mode in [0, 1, 2]:
                    for classifier_name in classifier_names:
                        accuracy, matrix, y_pred = evaluate_model(
                            X_train_prepared,
                            y_train_prepared,
                            X_test_prepared,
                            y_test,
                            classifier_name,
                            feature_mode,
                            seed=eval_seed
                        )

                        row = {
                            "eval_seed": eval_seed,
                            "noise_level": noise_level,
                            "strategy": strategy_name,
                            "classifier": classifier_name,
                            "feature_mode": FEATURE_MODES[feature_mode],
                            "accuracy": accuracy,
                            "train_size": len(y_train_prepared),
                        }

                        rows.append(row)

                        key = (
                            eval_seed,
                            noise_level,
                            strategy_name,
                            classifier_name,
                            FEATURE_MODES[feature_mode],
                        )
                        details[key] = (accuracy, matrix, y_test, y_pred)

    results_df = pd.DataFrame(rows)

    return results_df, details


# ============================================================
# 8. АНАЛИЗ РЕЗУЛЬТАТОВ
# ============================================================

def make_summary_tables(results_df):
    summary_by_noise = (
        results_df
        .groupby(["noise_level", "strategy", "classifier", "feature_mode"], as_index=False)
        .agg(
            mean_accuracy=("accuracy", "mean"),
            std_accuracy=("accuracy", "std"),
            min_accuracy=("accuracy", "min"),
            max_accuracy=("accuracy", "max"),
            mean_train_size=("train_size", "mean"),
        )
        .sort_values(["noise_level", "mean_accuracy"], ascending=[True, False])
    )

    # Только шумные уровни: 1%, 3%, 5%, 10%.
    noisy_only = results_df[results_df["noise_level"] > 0].copy()

    summary_noisy_overall = (
        noisy_only
        .groupby(["strategy", "classifier", "feature_mode"], as_index=False)
        .agg(
            mean_accuracy=("accuracy", "mean"),
            std_accuracy=("accuracy", "std"),
            min_accuracy=("accuracy", "min"),
            max_accuracy=("accuracy", "max"),
            mean_train_size=("train_size", "mean"),
        )
        .sort_values("mean_accuracy", ascending=False)
    )

    # Worst-case ориентир: выбираем конфигурации, у которых хороший минимум.
    summary_worst_case = summary_noisy_overall.sort_values(
        ["min_accuracy", "mean_accuracy"],
        ascending=[False, False]
    )

    best_by_noise = (
        summary_by_noise
        .groupby("noise_level", as_index=False)
        .head(10)
        .sort_values(["noise_level", "mean_accuracy"], ascending=[True, False])
    )

    return summary_by_noise, summary_noisy_overall, summary_worst_case, best_by_noise


def print_detailed_best_10_percent(results_df, details):
    """
    Печатает confusion matrix для лучшей конфигурации при 10% шуме
    по среднему accuracy.
    """
    ten = results_df[results_df["noise_level"] == 0.10].copy()

    grouped = (
        ten
        .groupby(["strategy", "classifier", "feature_mode"], as_index=False)
        .agg(mean_accuracy=("accuracy", "mean"))
        .sort_values("mean_accuracy", ascending=False)
    )

    best = grouped.iloc[0]

    print()
    print("=" * 80)
    print("Лучшая конфигурация при 10% шуме по mean accuracy:")
    print("=" * 80)
    display(best.to_frame().T)

    # Берем первый eval_seed для detailed output.
    eval_seed = EVAL_SEEDS[0]
    key = (
        eval_seed,
        0.10,
        best["strategy"],
        best["classifier"],
        best["feature_mode"],
    )

    if key not in details:
        print("Не удалось найти detailed result для ключа:", key)
        return

    accuracy, matrix, y_test, y_pred = details[key]

    print()
    print(f"Detailed result for seed={eval_seed}")
    print(f"Strategy: {best['strategy']}")
    print(f"Classifier: {best['classifier']}")
    print(f"Feature mode: {best['feature_mode']}")
    print(f"Accuracy: {accuracy:.3f}")
    print()
    print("Confusion matrix:")
    print(matrix)
    print()
    print("Classification report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=[CLASS_NAMES[0], CLASS_NAMES[1], CLASS_NAMES[2]]
    ))


# ============================================================
# 9. ГРАФИКИ
# ============================================================

def plot_top_configurations_by_noise(summary_by_noise):
    """
    График: лучшая средняя accuracy для каждого уровня шума.
    """
    best_one = (
        summary_by_noise
        .groupby("noise_level", as_index=False)
        .head(1)
        .copy()
    )

    labels = []
    for _, row in best_one.iterrows():
        label = (
            f"{row['strategy']} | "
            f"{row['classifier']} | "
            f"{row['feature_mode']}"
        )
        labels.append(label)

    plt.figure(figsize=(9, 5))
    plt.plot(
        best_one["noise_level"] * 100,
        best_one["mean_accuracy"],
        marker="o"
    )

    for x, y, label in zip(best_one["noise_level"] * 100, best_one["mean_accuracy"], labels):
        plt.text(x, y + 0.015, f"{y:.3f}", ha="center", fontsize=9)

    plt.title("Лучшая mean accuracy для каждого уровня шума")
    plt.xlabel("Уровень шума, %")
    plt.ylabel("Mean accuracy over seeds")
    plt.ylim(0.0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.savefig("stage7_3_best_accuracy_by_noise.png", dpi=200, bbox_inches="tight")
    plt.show()

    print("Сохранен график: stage7_3_best_accuracy_by_noise.png")


def plot_selected_strategy_comparison(summary_by_noise):
    """
    Сравниваем несколько ключевых стратегий для Decision Tree и times/times+d1.
    """
    selected = summary_by_noise[
        (summary_by_noise["classifier"] == "Decision Tree") &
        (
            (summary_by_noise["strategy"].isin([
                "clean_train",
                "matched_noisy_train",
                "matched_noisy_train_filtered",
                "aug_fixed_1_3_5_10",
                "aug_fixed_1_3_5_10_filtered",
                "aug_random_0_10",
                "aug_random_0_10_filtered",
            ]))
        ) &
        (summary_by_noise["feature_mode"].isin(["times", "times + d1"]))
    ].copy()

    plt.figure(figsize=(11, 6))

    for (strategy, feature_mode), group in selected.groupby(["strategy", "feature_mode"]):
        if strategy in ["clean_train", "matched_noisy_train", "matched_noisy_train_filtered", "aug_fixed_1_3_5_10_filtered"]:
            label = f"{strategy} | {feature_mode}"
            plt.plot(
                group["noise_level"] * 100,
                group["mean_accuracy"],
                marker="o",
                label=label
            )

    plt.title("Decision Tree: сравнение ключевых стратегий")
    plt.xlabel("Уровень шума, %")
    plt.ylabel("Mean accuracy over seeds")
    plt.ylim(0.0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    plt.savefig("stage7_3_decision_tree_strategy_comparison.png", dpi=200, bbox_inches="tight")
    plt.show()

    print("Сохранен график: stage7_3_decision_tree_strategy_comparison.png")


def plot_heatmap_for_best_classifier(summary_noisy_overall):
    """
    Текстовая/матричная визуализация mean accuracy по стратегиям и feature modes
    для лучшего classifier.
    """
    # Берем лучший классификатор по среднему.
    best_classifier = (
        summary_noisy_overall
        .groupby("classifier", as_index=False)
        .agg(mean_accuracy=("mean_accuracy", "mean"))
        .sort_values("mean_accuracy", ascending=False)
        .iloc[0]["classifier"]
    )

    subset = summary_noisy_overall[summary_noisy_overall["classifier"] == best_classifier].copy()

    pivot = subset.pivot_table(
        index="strategy",
        columns="feature_mode",
        values="mean_accuracy",
        aggfunc="mean"
    )

    plt.figure(figsize=(8, 5))
    im = plt.imshow(pivot.values, aspect="auto")
    plt.colorbar(im, label="Mean accuracy over noisy levels")
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=30, ha="right")
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.title(f"Mean accuracy heatmap: {best_classifier}")

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            value = pivot.values[i, j]
            plt.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=8)

    plt.tight_layout()
    plt.savefig("stage7_3_heatmap_best_classifier.png", dpi=200, bbox_inches="tight")
    plt.show()

    print("Сохранен график: stage7_3_heatmap_best_classifier.png")


# ============================================================
# 10. ЗАПУСК ЭТАПА 7.3
# ============================================================

print("=" * 80)
print("ЭТАП 7.3: РАСШИРЕННАЯ ПРОВЕРКА УСТОЙЧИВОСТИ К ШУМУ")
print("=" * 80)

print()
print("Генерируем чистый датасет...")
X_clean, y = generate_dataset(N_PER_CLASS, GENERATION_SEED)

print()
check_dataset(X_clean, y)

clean_df = pd.DataFrame(X_clean, columns=[f"t{i}" for i in range(1, X_clean.shape[1] + 1)])
clean_df["class"] = y
clean_df.to_csv("stage7_3_clean_dataset.csv", index=False)
print("Чистый датасет сохранен: stage7_3_clean_dataset.csv")

print()
print("Запускаем расширенный эксперимент...")
results_df, details = run_stage_7_2_experiment(X_clean, y)
results_df.to_csv("stage7_3_all_results.csv", index=False)

print()
print("=" * 80)
print("Первые строки общей таблицы результатов")
print("=" * 80)
display(results_df.head())

print()
print("=" * 80)
print("Размер общей таблицы результатов")
print("=" * 80)
print(results_df.shape)

summary_by_noise, summary_noisy_overall, summary_worst_case, best_by_noise = make_summary_tables(results_df)

summary_by_noise.to_csv("stage7_3_summary_by_noise.csv", index=False)
summary_noisy_overall.to_csv("stage7_3_summary_noisy_overall.csv", index=False)
summary_worst_case.to_csv("stage7_3_summary_worst_case.csv", index=False)
best_by_noise.to_csv("stage7_3_best_by_noise.csv", index=False)

print()
print("=" * 80)
print("ТОП-10 конфигураций для каждого уровня шума")
print("=" * 80)
display(best_by_noise)

print()
print("=" * 80)
print("Лучшие конфигурации по средней accuracy на шумных уровнях 1/3/5/10%")
print("=" * 80)
display(summary_noisy_overall.head(20))

print()
print("=" * 80)
print("Лучшие конфигурации по worst-case accuracy на шумных уровнях")
print("=" * 80)
display(summary_worst_case.head(20))

print_detailed_best_10_percent(results_df, details)

print()
print("Строим графики...")
plot_top_configurations_by_noise(summary_by_noise)
plot_selected_strategy_comparison(summary_by_noise)
plot_heatmap_for_best_classifier(summary_noisy_overall)

print()
print("Готово. Основные файлы:")
print("stage7_3_clean_dataset.csv")
print("stage7_3_all_results.csv")
print("stage7_3_summary_by_noise.csv")
print("stage7_3_summary_noisy_overall.csv")
print("stage7_3_summary_worst_case.csv")
print("stage7_3_best_by_noise.csv")
print("stage7_3_best_accuracy_by_noise.png")
print("stage7_3_decision_tree_strategy_comparison.png")
print("stage7_3_heatmap_best_classifier.png")

## Этап 8 — MCMC-инверсия профиля скорости

Этап 8 CHECKED:
MCMC-инверсия параметров вертикального профиля скорости по travel-time curve.

Этот ноутбук является проверенной версией этапа 8.
Что улучшено относительно первой быстрой версии:
1. Добавлены подробные комментарии на русском языке.
2. Добавлен короткий random search перед MCMC, чтобы стартовать не из случайно плохой точки.
3. Добавлены RMSE в миллисекундах и итоговая таблица качества восстановления.
4. В отчете сохраняются и best estimate, и posterior median/mean после burn-in.
5. Явно указаны ограничения: это не полноценная промышленная инверсия, а учебная параметрическая обратная задача.
6. Сетка и число итераций выбраны так, чтобы Colab считал разумное время.

Логика:
- Прямая задача: параметры профиля -> travel-time curve.
- Обратная задача: travel-time curve -> параметры профиля.

Часть 8.1:
Двухслойная модель: v1, v2, h.

Часть 8.2:
Сигмоидальный профиль: v_top, v_bottom, z0, k.

In [ ]:
import heapq
import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

# FAST — быстрее для первого запуска в Colab.
# ACCURATE — дольше, но стабильнее.
RUN_MODE = "FAST"  # можно заменить на "ACCURATE"

@dataclass
class ModelConfig:
    x_min: float = 0.0
    x_max: float = 1200.0
    z_min: float = 0.0
    z_max: float = 500.0
    nx: int = 41
    nz: int = 31
    source_x: float = 600.0
    source_z: float = 400.0
    receiver_count: int = 17
    receiver_z: float = 0.0


if RUN_MODE == "ACCURATE":
    CONFIG = ModelConfig(nx=51, nz=41, receiver_count=21)
    N_RANDOM_SEARCH_TWO_LAYER = 120
    N_RANDOM_SEARCH_SIGMOID = 180
    N_ITER_TWO_LAYER = 1400
    N_ITER_SIGMOID = 1800
else:
    CONFIG = ModelConfig(nx=41, nz=31, receiver_count=17)
    N_RANDOM_SEARCH_TWO_LAYER = 60
    N_RANDOM_SEARCH_SIGMOID = 90
    N_ITER_TWO_LAYER = 700
    N_ITER_SIGMOID = 900

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

BURN_IN_FRACTION = 0.35
OBSERVATION_NOISE_LEVEL = 0.005  # 0.5% шум. Для чистого теста можно поставить 0.0

print("RUN_MODE:", RUN_MODE)
print("Grid nx,nz:", CONFIG.nx, CONFIG.nz)
print("Receivers:", CONFIG.receiver_count)
print("Two-layer iterations:", N_ITER_TWO_LAYER)
print("Sigmoid iterations:", N_ITER_SIGMOID)


# ============================================================
# 2. СЕТКА И ПРЯМАЯ ЗАДАЧА
# ============================================================

XS = np.linspace(CONFIG.x_min, CONFIG.x_max, CONFIG.nx)
ZS = np.linspace(CONFIG.z_min, CONFIG.z_max, CONFIG.nz)
DX = XS[1] - XS[0]
DZ = ZS[1] - ZS[0]

SOURCE_IX = int(np.argmin(np.abs(XS - CONFIG.source_x)))
SOURCE_IZ = int(np.argmin(np.abs(ZS - CONFIG.source_z)))

RECEIVER_XS = np.linspace(
    CONFIG.x_min + 50.0,
    CONFIG.x_max - 50.0,
    CONFIG.receiver_count
)
RECEIVER_IZ = int(np.argmin(np.abs(ZS - CONFIG.receiver_z)))
RECEIVER_IXS = [int(np.argmin(np.abs(XS - x))) for x in RECEIVER_XS]


def build_velocity_model(profile_func):
    """
    Строит 2D-модель скорости.
    В этой постановке скорость зависит только от глубины z, поэтому профиль вертикальный.
    """
    velocity_model = np.zeros((len(ZS), len(XS)))
    for iz, z in enumerate(ZS):
        velocity_model[iz, :] = profile_func(z)
    return velocity_model


def dijkstra_travel_times(velocity_model):
    """
    Прямая задача.

    Сетка рассматривается как граф.
    Каждый узел соединен с 8 соседями.
    Вес ребра = расстояние / средняя скорость на ребре.
    Алгоритм Дейкстры ищет путь минимального времени от источника до всех узлов.

    Это не полноценный промышленный Fast Marching Method, но близкий учебный графовый аналог
    для расчета travel-time curve на простой сетке.
    """
    nz, nx = velocity_model.shape
    travel_times = np.full((nz, nx), np.inf)
    travel_times[SOURCE_IZ, SOURCE_IX] = 0.0

    neighbors = [
        (-1, 0), (1, 0), (0, -1), (0, 1),
        (-1, -1), (-1, 1), (1, -1), (1, 1),
    ]

    heap = [(0.0, SOURCE_IZ, SOURCE_IX)]

    while heap:
        current_time, iz, ix = heapq.heappop(heap)
        if current_time > travel_times[iz, ix]:
            continue

        for dzi, dxi in neighbors:
            next_iz = iz + dzi
            next_ix = ix + dxi

            if not (0 <= next_iz < nz and 0 <= next_ix < nx):
                continue

            step_distance = math.sqrt((dxi * DX) ** 2 + (dzi * DZ) ** 2)
            average_velocity = 0.5 * (
                velocity_model[iz, ix] + velocity_model[next_iz, next_ix]
            )
            new_time = current_time + step_distance / average_velocity

            if new_time < travel_times[next_iz, next_ix]:
                travel_times[next_iz, next_ix] = new_time
                heapq.heappush(heap, (new_time, next_iz, next_ix))

    return np.array([travel_times[RECEIVER_IZ, ix] for ix in RECEIVER_IXS])


def simulate_curve(profile_func):
    return dijkstra_travel_times(build_velocity_model(profile_func))


def add_relative_noise(curve, noise_level, seed):
    if noise_level == 0:
        return curve.copy()
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, noise_level * curve, size=curve.shape)
    return np.maximum(curve + noise, 1e-6)


# ============================================================
# 3. ПРОФИЛИ СКОРОСТИ
# ============================================================


def two_layer_profile(v1, v2, h):
    """
    Двухслойный профиль:
    v1 — скорость верхнего слоя;
    v2 — скорость нижнего слоя;
    h — глубина границы.
    """
    def profile(z):
        return v1 if z < h else v2
    return profile


def sigmoid_profile(v_top, v_bottom, z0, k):
    """
    Сигмоидальный профиль скорости:

    v(z) = v_top + (v_bottom - v_top) / (1 + exp(-k * (z - z0)))

    v_top    — скорость в верхней части;
    v_bottom — скорость в нижней части;
    z0       — глубина перехода;
    k        — резкость перехода.

    Малый k дает плавный переход.
    Большой k дает резкий переход, похожий на границу слоев.
    """
    def profile(z):
        return v_top + (v_bottom - v_top) / (1.0 + math.exp(-k * (z - z0)))
    return profile


def profile_values(profile_func):
    return np.array([profile_func(z) for z in ZS])


# ============================================================
# 4. LOSS, RANDOM SEARCH И MCMC
# ============================================================


def curve_loss(observed_curve, predicted_curve, sigma):
    residual = (observed_curve - predicted_curve) / sigma
    return float(np.sum(residual ** 2))


def curve_rmse_ms(observed_curve, predicted_curve):
    return float(np.sqrt(np.mean((observed_curve - predicted_curve) ** 2)) * 1000.0)


def reflect_to_bounds(params, bounds):
    reflected = []
    for value, (low, high) in zip(params, bounds):
        if value < low:
            value = low + (low - value)
        if value > high:
            value = high - (value - high)
        value = min(max(value, low), high)
        reflected.append(value)
    return np.array(reflected, dtype=float)


def sample_uniform(bounds, rng):
    return np.array([rng.uniform(low, high) for low, high in bounds], dtype=float)


def random_search_initialization(
    observed_curve,
    make_profile,
    bounds,
    sigma,
    n_samples,
    seed,
    label,
):
    """
    Небольшой предварительный random search.
    Он нужен, чтобы MCMC начинался не из совсем плохой точки.
    """
    rng = np.random.default_rng(seed)
    best_params = None
    best_curve = None
    best_loss = np.inf

    for i in range(n_samples):
        params = sample_uniform(bounds, rng)
        curve = simulate_curve(make_profile(params))
        loss = curve_loss(observed_curve, curve, sigma)

        if loss < best_loss:
            best_loss = loss
            best_params = params
            best_curve = curve

    print(f"{label}: random search best loss = {best_loss:.2f}")
    print(f"{label}: random search best params = {best_params}")
    return best_params, best_curve, best_loss


def run_mcmc(
    observed_curve,
    make_profile,
    initial_params,
    bounds,
    proposal_scales,
    n_iter,
    sigma,
    seed,
    label,
):
    """
    Простой Metropolis-Hastings MCMC.

    Идея:
    - если новые параметры улучшают совпадение curve, принимаем их;
    - если ухудшают, иногда тоже принимаем, чтобы не застрять в локальном минимуме.
    """
    rng = np.random.default_rng(seed)

    current_params = reflect_to_bounds(np.array(initial_params, dtype=float), bounds)
    current_curve = simulate_curve(make_profile(current_params))
    current_loss = curve_loss(observed_curve, current_curve, sigma)

    chain = np.zeros((n_iter, len(current_params)))
    losses = np.zeros(n_iter)
    accepted = np.zeros(n_iter, dtype=bool)

    best_params = current_params.copy()
    best_curve = current_curve.copy()
    best_loss = current_loss

    for i in range(n_iter):
        proposal = current_params + rng.normal(0.0, proposal_scales, size=len(current_params))
        proposal = reflect_to_bounds(proposal, bounds)

        proposal_curve = simulate_curve(make_profile(proposal))
        proposal_loss = curve_loss(observed_curve, proposal_curve, sigma)

        delta = proposal_loss - current_loss
        if delta < 0:
            accept = True
        else:
            # При очень большом delta exp(-0.5*delta) становится почти 0 — это нормально.
            accept_probability = math.exp(-0.5 * delta) if delta < 1400 else 0.0
            accept = rng.random() < accept_probability

        if accept:
            current_params = proposal
            current_curve = proposal_curve
            current_loss = proposal_loss
            accepted[i] = True

        if current_loss < best_loss:
            best_params = current_params.copy()
            best_curve = current_curve.copy()
            best_loss = current_loss

        chain[i, :] = current_params
        losses[i] = current_loss

        if (i + 1) % max(1, n_iter // 10) == 0:
            print(
                f"{label}: {i + 1}/{n_iter}, "
                f"current loss={current_loss:.2f}, best loss={best_loss:.2f}, "
                f"acceptance={accepted[:i+1].mean():.3f}"
            )

    return {
        "chain": chain,
        "losses": losses,
        "accepted": accepted,
        "best_params": best_params,
        "best_curve": best_curve,
        "best_loss": best_loss,
        "acceptance_rate": float(accepted.mean()),
    }


def summarize_chain(chain, param_names):
    burn_in = int(len(chain) * BURN_IN_FRACTION)
    post_chain = chain[burn_in:]
    rows = []

    for i, name in enumerate(param_names):
        values = post_chain[:, i]
        rows.append({
            "parameter": name,
            "posterior_mean": float(np.mean(values)),
            "posterior_median": float(np.median(values)),
            "posterior_std": float(np.std(values)),
            "q05": float(np.quantile(values, 0.05)),
            "q95": float(np.quantile(values, 0.95)),
        })

    return pd.DataFrame(rows), post_chain


# ============================================================
# 5. ГРАФИКИ
# ============================================================


def plot_profile_comparison(true_profile, estimated_profile, title, filename):
    plt.figure(figsize=(6, 6))
    plt.plot(profile_values(true_profile), ZS, label="Истинный профиль")
    plt.plot(profile_values(estimated_profile), ZS, "--", label="Восстановленный профиль")
    plt.gca().invert_yaxis()
    plt.xlabel("Скорость, м/с")
    plt.ylabel("Глубина z, м")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print("Сохранен график:", filename)


def plot_curve_comparison(clean_curve, observed_curve, predicted_curve, title, filename):
    plt.figure(figsize=(8, 5))
    plt.plot(RECEIVER_XS, clean_curve, marker="o", label="Истинная чистая curve")
    plt.plot(RECEIVER_XS, observed_curve, marker="o", label="Наблюдаемая curve")
    plt.plot(RECEIVER_XS, predicted_curve, marker="o", linestyle="--", label="Расчетная curve по восстановленным параметрам")
    plt.xlabel("x приемника, м")
    plt.ylabel("Время прихода, с")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print("Сохранен график:", filename)


def plot_loss_history(losses, title, filename):
    plt.figure(figsize=(8, 5))
    plt.plot(losses)
    plt.xlabel("Итерация MCMC")
    plt.ylabel("Loss")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print("Сохранен график:", filename)


def plot_parameter_traces(chain, param_names, true_params, filename_prefix):
    for i, name in enumerate(param_names):
        plt.figure(figsize=(8, 4))
        plt.plot(chain[:, i])
        plt.axhline(true_params[i], linestyle="--", label=f"Истинное {name}")
        plt.xlabel("Итерация MCMC")
        plt.ylabel(name)
        plt.title(f"История параметра {name}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        filename = f"{filename_prefix}_trace_{name}.png"
        plt.savefig(filename, dpi=200, bbox_inches="tight")
        plt.show()
        print("Сохранен график:", filename)


def plot_parameter_histograms(post_chain, param_names, true_params, filename_prefix):
    for i, name in enumerate(param_names):
        plt.figure(figsize=(7, 4))
        plt.hist(post_chain[:, i], bins=30, alpha=0.8)
        plt.axvline(true_params[i], linestyle="--", label=f"Истинное {name}")
        plt.xlabel(name)
        plt.ylabel("Частота")
        plt.title(f"Распределение параметра {name} после burn-in")
        plt.grid(True, alpha=0.3)
        plt.legend()
        filename = f"{filename_prefix}_hist_{name}.png"
        plt.savefig(filename, dpi=200, bbox_inches="tight")
        plt.show()
        print("Сохранен график:", filename)


# ============================================================
# 6. ЭТАП 8.1 — ДВУХСЛОЙНАЯ МОДЕЛЬ
# ============================================================

print("=" * 80)
print("ЭТАП 8.1 — MCMC ДЛЯ ДВУХСЛОЙНОЙ МОДЕЛИ")
print("=" * 80)

true_two_params = np.array([1800.0, 4600.0, 240.0])
true_two_profile = two_layer_profile(*true_two_params)
clean_two_curve = simulate_curve(true_two_profile)
observed_two_curve = add_relative_noise(clean_two_curve, OBSERVATION_NOISE_LEVEL, RANDOM_SEED + 10)
sigma_two = np.maximum(OBSERVATION_NOISE_LEVEL * observed_two_curve, 0.0015)


def make_two_layer(params):
    return two_layer_profile(params[0], params[1], params[2])


two_bounds = [
    (1300.0, 2600.0),
    (3000.0, 7000.0),
    (80.0, 420.0),
]

two_initial, _, _ = random_search_initialization(
    observed_curve=observed_two_curve,
    make_profile=make_two_layer,
    bounds=two_bounds,
    sigma=sigma_two,
    n_samples=N_RANDOM_SEARCH_TWO_LAYER,
    seed=RANDOM_SEED + 101,
    label="two-layer",
)

result_two = run_mcmc(
    observed_curve=observed_two_curve,
    make_profile=make_two_layer,
    initial_params=two_initial,
    bounds=two_bounds,
    proposal_scales=np.array([55.0, 130.0, 14.0]),
    n_iter=N_ITER_TWO_LAYER,
    sigma=sigma_two,
    seed=RANDOM_SEED + 201,
    label="two-layer",
)

best_two_params = result_two["best_params"]
best_two_profile = make_two_layer(best_two_params)

two_summary, two_post_chain = summarize_chain(result_two["chain"], ["v1", "v2", "h"])

two_chain_df = pd.DataFrame(result_two["chain"], columns=["v1", "v2", "h"])
two_chain_df["loss"] = result_two["losses"]
two_chain_df["accepted"] = result_two["accepted"]

two_summary.to_csv("stage8_checked_two_layer_posterior_summary.csv", index=False)
two_chain_df.to_csv("stage8_checked_two_layer_chain.csv", index=False)

print("Истинные параметры [v1, v2, h]:", true_two_params)
print("Лучшие параметры MCMC:", best_two_params)
print("Acceptance rate:", result_two["acceptance_rate"])
print("Best curve RMSE, ms:", curve_rmse_ms(observed_two_curve, result_two["best_curve"]))
print("Posterior summary:")
display(two_summary)

plot_profile_comparison(
    true_two_profile,
    best_two_profile,
    "Двухслойная модель: истинный и восстановленный профиль скорости",
    "stage8_checked_two_layer_profile_comparison.png",
)
plot_curve_comparison(
    clean_two_curve,
    observed_two_curve,
    result_two["best_curve"],
    "Двухслойная модель: observed и predicted travel-time curve",
    "stage8_checked_two_layer_curve_comparison.png",
)
plot_loss_history(
    result_two["losses"],
    "Двухслойная модель: история loss",
    "stage8_checked_two_layer_loss_history.png",
)
plot_parameter_traces(
    result_two["chain"],
    ["v1", "v2", "h"],
    true_two_params,
    "stage8_checked_two_layer",
)
plot_parameter_histograms(
    two_post_chain,
    ["v1", "v2", "h"],
    true_two_params,
    "stage8_checked_two_layer",
)


# ============================================================
# 7. ЭТАП 8.2 — СИГМОИДАЛЬНЫЙ ПРОФИЛЬ
# ============================================================

print("\n" + "=" * 80)
print("ЭТАП 8.2 — MCMC ДЛЯ СИГМОИДАЛЬНОГО ПРОФИЛЯ")
print("=" * 80)

true_sig_params = np.array([1500.0, 5200.0, 250.0, 0.030])
true_sig_profile = sigmoid_profile(*true_sig_params)
clean_sig_curve = simulate_curve(true_sig_profile)
observed_sig_curve = add_relative_noise(clean_sig_curve, OBSERVATION_NOISE_LEVEL, RANDOM_SEED + 20)
sigma_sig = np.maximum(OBSERVATION_NOISE_LEVEL * observed_sig_curve, 0.0015)


def make_sigmoid(params):
    return sigmoid_profile(params[0], params[1], params[2], params[3])


sig_bounds = [
    (1200.0, 2600.0),
    (3000.0, 7000.0),
    (80.0, 420.0),
    (0.005, 0.080),
]

sig_initial, _, _ = random_search_initialization(
    observed_curve=observed_sig_curve,
    make_profile=make_sigmoid,
    bounds=sig_bounds,
    sigma=sigma_sig,
    n_samples=N_RANDOM_SEARCH_SIGMOID,
    seed=RANDOM_SEED + 301,
    label="sigmoid",
)

result_sig = run_mcmc(
    observed_curve=observed_sig_curve,
    make_profile=make_sigmoid,
    initial_params=sig_initial,
    bounds=sig_bounds,
    proposal_scales=np.array([55.0, 150.0, 14.0, 0.003]),
    n_iter=N_ITER_SIGMOID,
    sigma=sigma_sig,
    seed=RANDOM_SEED + 401,
    label="sigmoid",
)

best_sig_params = result_sig["best_params"]
best_sig_profile = make_sigmoid(best_sig_params)

sig_summary, sig_post_chain = summarize_chain(result_sig["chain"], ["v_top", "v_bottom", "z0", "k"])

sig_chain_df = pd.DataFrame(result_sig["chain"], columns=["v_top", "v_bottom", "z0", "k"])
sig_chain_df["loss"] = result_sig["losses"]
sig_chain_df["accepted"] = result_sig["accepted"]

sig_summary.to_csv("stage8_checked_sigmoid_posterior_summary.csv", index=False)
sig_chain_df.to_csv("stage8_checked_sigmoid_chain.csv", index=False)

print("Истинные параметры [v_top, v_bottom, z0, k]:", true_sig_params)
print("Лучшие параметры MCMC:", best_sig_params)
print("Acceptance rate:", result_sig["acceptance_rate"])
print("Best curve RMSE, ms:", curve_rmse_ms(observed_sig_curve, result_sig["best_curve"]))
print("Posterior summary:")
display(sig_summary)

plot_profile_comparison(
    true_sig_profile,
    best_sig_profile,
    "Сигмоидальный профиль: истинный и восстановленный профиль скорости",
    "stage8_checked_sigmoid_profile_comparison.png",
)
plot_curve_comparison(
    clean_sig_curve,
    observed_sig_curve,
    result_sig["best_curve"],
    "Сигмоидальный профиль: observed и predicted travel-time curve",
    "stage8_checked_sigmoid_curve_comparison.png",
)
plot_loss_history(
    result_sig["losses"],
    "Сигмоидальный профиль: история loss",
    "stage8_checked_sigmoid_loss_history.png",
)
plot_parameter_traces(
    result_sig["chain"],
    ["v_top", "v_bottom", "z0", "k"],
    true_sig_params,
    "stage8_checked_sigmoid",
)
plot_parameter_histograms(
    sig_post_chain,
    ["v_top", "v_bottom", "z0", "k"],
    true_sig_params,
    "stage8_checked_sigmoid",
)


# ============================================================
# 8. ИТОГОВАЯ ТАБЛИЦА
# ============================================================

final_rows = []

for model_name, names, true_params, best_params, posterior_summary, observed_curve, best_curve in [
    (
        "two_layer",
        ["v1", "v2", "h"],
        true_two_params,
        best_two_params,
        two_summary,
        observed_two_curve,
        result_two["best_curve"],
    ),
    (
        "sigmoid",
        ["v_top", "v_bottom", "z0", "k"],
        true_sig_params,
        best_sig_params,
        sig_summary,
        observed_sig_curve,
        result_sig["best_curve"],
    ),
]:
    rmse_ms = curve_rmse_ms(observed_curve, best_curve)
    for i, name in enumerate(names):
        post_row = posterior_summary[posterior_summary["parameter"] == name].iloc[0]
        final_rows.append({
            "model": model_name,
            "parameter": name,
            "true_value": true_params[i],
            "best_estimate": best_params[i],
            "posterior_median": post_row["posterior_median"],
            "absolute_error_best": abs(best_params[i] - true_params[i]),
            "curve_rmse_ms": rmse_ms,
        })

final_summary = pd.DataFrame(final_rows)
print("\n" + "=" * 80)
print("ИТОГОВАЯ СВОДКА ЭТАПА 8")
print("=" * 80)
display(final_summary)
final_summary.to_csv("stage8_checked_final_parameter_recovery_summary.csv", index=False)

print("\nГотово. Основные файлы сохранены с префиксом stage8_checked_*.png / *.csv")

## Этап 9 — регрессионное начальное приближение

Этап 9 CHECKED:
обучаемая регрессионная модель для восстановления параметров сигмоидального профиля.

Почему этот этап нужен:
MCMC сам по себе не обучается. Он перебирает параметры и принимает/отклоняет
новые варианты. Чтобы получить более сильное начальное приближение, добавляем
обучаемую модель-регрессор:

    travel-time curve -> параметры профиля скорости

После регрессора можно делать локальное уточнение параметров или запускать MCMC
уже не "вслепую", а из хорошей начальной точки.

Что исправлено по сравнению с первой версией stage9:
1. Нет утечки данных:
   сначала делим clean dataset на train/test, затем создаем шумовые копии только train.
2. Тестирование проводится на нескольких уровнях шума:
   0%, 0.5%, 1%, 2%, 5%.
3. Сравниваются две модели:
   RandomForestRegressor и ExtraTreesRegressor.
4. Ошибки считаются отдельно по каждому параметру:
   v_top, v_bottom, z0, k.
5. Дополнительно считается curve RMSE в миллисекундах:
   насколько travel-time curve, полученная из предсказанных параметров,
   совпадает с истинной clean curve.
6. Лучшая модель выбирается не по одному демо-примеру, а по средней ошибке
   на всех тестовых шумовых уровнях.
7. Для демо-примера используется multi-start local refinement:
   регрессор дает начальное приближение, затем несколько локальных запусков
   уточняют параметры по ошибке travel-time curve.
8. Все результаты сохраняются с префиксом stage9_checked_.

Важно:
регрессор не доказывает единственность решения обратной задачи.
Он дает быстрый point estimate параметров. Если разные параметры дают похожие
travel-time curves, точные параметры могут восстанавливаться не идеально,
даже если curve RMSE маленький.

In [ ]:
import heapq
import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

RUN_MODE = "ACCURATE"  # "FAST" или "ACCURATE"

if RUN_MODE == "FAST":
    N_BASE_SAMPLES = 2000
    N_ESTIMATORS = 300
    MAX_CURVE_RMSE_TEST = 100
else:
    N_BASE_SAMPLES = 6000
    N_ESTIMATORS = 600
    MAX_CURVE_RMSE_TEST = 250

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Шумовые копии для обучения.
TRAIN_NOISE_LEVELS = [0.0, 0.005, 0.01, 0.02, 0.05]

# Уровни шума для теста.
TEST_NOISE_LEVELS = [0.0, 0.005, 0.01, 0.02, 0.05]

# Шум в демо observed curve.
DEMO_OBS_NOISE = 0.01


@dataclass
class ModelConfig:
    x_min: float = 0.0
    x_max: float = 1200.0
    z_min: float = 0.0
    z_max: float = 500.0

    nx: int = 51
    nz: int = 41

    source_x: float = 600.0
    source_z: float = 400.0

    receiver_count: int = 21
    receiver_z: float = 0.0


CONFIG = ModelConfig()

PARAM_NAMES = ["v_top", "v_bottom", "z0", "k"]

PARAM_BOUNDS = {
    "v_top": (1200.0, 2600.0),
    "v_bottom": (3000.0, 7000.0),
    "z0": (80.0, 420.0),
    "k": (0.005, 0.080),
}

PARAM_RANGES = np.array([
    PARAM_BOUNDS["v_top"][1] - PARAM_BOUNDS["v_top"][0],
    PARAM_BOUNDS["v_bottom"][1] - PARAM_BOUNDS["v_bottom"][0],
    PARAM_BOUNDS["z0"][1] - PARAM_BOUNDS["z0"][0],
    PARAM_BOUNDS["k"][1] - PARAM_BOUNDS["k"][0],
])


# ============================================================
# 2. СЕТКА И ПРЯМАЯ МОДЕЛЬ
# ============================================================

def build_grid(config):
    xs = np.linspace(config.x_min, config.x_max, config.nx)
    zs = np.linspace(config.z_min, config.z_max, config.nz)
    dx = xs[1] - xs[0]
    dz = zs[1] - zs[0]
    return xs, zs, dx, dz


XS, ZS, DX, DZ = build_grid(CONFIG)

SOURCE_IX = int(np.argmin(np.abs(XS - CONFIG.source_x)))
SOURCE_IZ = int(np.argmin(np.abs(ZS - CONFIG.source_z)))

RECEIVER_XS = np.linspace(
    CONFIG.x_min + 50.0,
    CONFIG.x_max - 50.0,
    CONFIG.receiver_count
)

RECEIVER_IZ = int(np.argmin(np.abs(ZS - CONFIG.receiver_z)))
RECEIVER_IXS = [int(np.argmin(np.abs(XS - x))) for x in RECEIVER_XS]


def sigmoid_profile(v_top, v_bottom, z0, k):
    def profile(z):
        return v_top + (v_bottom - v_top) / (1.0 + math.exp(-k * (z - z0)))
    return profile


def profile_values_from_params(params):
    v_top, v_bottom, z0, k = params
    profile = sigmoid_profile(v_top, v_bottom, z0, k)
    return np.array([profile(z) for z in ZS])


def build_velocity_model_from_params(params):
    velocity_model = np.zeros((len(ZS), len(XS)))
    profile_values = profile_values_from_params(params)

    for iz, velocity in enumerate(profile_values):
        velocity_model[iz, :] = velocity

    return velocity_model


def dijkstra_travel_times(velocity_model):
    nz, nx = velocity_model.shape

    travel_times = np.full((nz, nx), np.inf)
    travel_times[SOURCE_IZ, SOURCE_IX] = 0.0

    neighbors = [
        (-1, 0), (1, 0), (0, -1), (0, 1),
        (-1, -1), (-1, 1), (1, -1), (1, 1),
    ]

    heap = [(0.0, SOURCE_IZ, SOURCE_IX)]

    while heap:
        current_time, iz, ix = heapq.heappop(heap)

        if current_time > travel_times[iz, ix]:
            continue

        for dzi, dxi in neighbors:
            next_iz = iz + dzi
            next_ix = ix + dxi

            if not (0 <= next_iz < nz and 0 <= next_ix < nx):
                continue

            step_distance = math.sqrt((dxi * DX) ** 2 + (dzi * DZ) ** 2)
            average_velocity = 0.5 * (
                velocity_model[iz, ix] + velocity_model[next_iz, next_ix]
            )

            step_time = step_distance / average_velocity
            new_time = current_time + step_time

            if new_time < travel_times[next_iz, next_ix]:
                travel_times[next_iz, next_ix] = new_time
                heapq.heappush(heap, (new_time, next_iz, next_ix))

    receiver_times = np.array([
        travel_times[RECEIVER_IZ, receiver_ix]
        for receiver_ix in RECEIVER_IXS
    ])

    return receiver_times


def simulate_curve(params):
    velocity_model = build_velocity_model_from_params(params)
    return dijkstra_travel_times(velocity_model)


# ============================================================
# 3. ДАТАСЕТ
# ============================================================

def sample_sigmoid_params(rng):
    v_top = rng.uniform(*PARAM_BOUNDS["v_top"])
    v_bottom = rng.uniform(*PARAM_BOUNDS["v_bottom"])
    z0 = rng.uniform(*PARAM_BOUNDS["z0"])
    k = rng.uniform(*PARAM_BOUNDS["k"])

    return np.array([v_top, v_bottom, z0, k], dtype=float)


def generate_regression_dataset(n_samples, seed):
    rng = np.random.default_rng(seed)

    X_curves = []
    y_params = []

    for i in range(n_samples):
        params = sample_sigmoid_params(rng)
        curve = simulate_curve(params)

        X_curves.append(curve)
        y_params.append(params)

        if (i + 1) % 100 == 0:
            print(f"Сгенерировано {i + 1}/{n_samples} примеров...")

    return np.array(X_curves), np.array(y_params)


def add_relative_noise(X, noise_level, seed):
    if noise_level == 0:
        return X.copy()

    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, noise_level * X, size=X.shape)

    return np.maximum(X + noise, 1e-6)


def make_noisy_training_copies(X_train, y_train, noise_levels, seed):
    X_parts = []
    y_parts = []

    for i, noise_level in enumerate(noise_levels):
        X_noisy = add_relative_noise(
            X_train,
            noise_level=noise_level,
            seed=seed + i * 1000 + int(noise_level * 10000)
        )
        X_parts.append(X_noisy)
        y_parts.append(y_train.copy())

    return np.vstack(X_parts), np.vstack(y_parts)


# ============================================================
# 4. ПРИЗНАКИ
# ============================================================

def build_regression_features(X):
    """
    Признаки:
    1. raw times;
    2. normalized curve shape;
    3. first derivative d1;
    4. robust summary features.

    Вторую производную d2 здесь сознательно не используем:
    на предыдущих этапах было видно, что d2 очень чувствительна к шуму.
    """
    X = np.asarray(X)

    raw = X

    x_min = X.min(axis=1, keepdims=True)
    x_range = X.max(axis=1, keepdims=True) - X.min(axis=1, keepdims=True)
    x_range = np.maximum(x_range, 1e-9)
    normalized = (X - x_min) / x_range

    d1 = np.gradient(X, axis=1)

    center_idx = X.shape[1] // 2
    left_slope = X[:, center_idx] - X[:, 0]
    right_slope = X[:, -1] - X[:, center_idx]

    summary = np.column_stack([
        X.min(axis=1),
        X.max(axis=1),
        X.mean(axis=1),
        X.std(axis=1),
        X.max(axis=1) - X.min(axis=1),
        X[:, center_idx],
        left_slope,
        right_slope,
    ])

    return np.hstack([raw, normalized, d1, summary])


def make_feature_names(n_receivers):
    names = []

    for i in range(n_receivers):
        names.append(f"t{i+1}")

    for i in range(n_receivers):
        names.append(f"t_norm{i+1}")

    for i in range(n_receivers):
        names.append(f"d1_{i+1}")

    names += [
        "t_min",
        "t_max",
        "t_mean",
        "t_std",
        "t_range",
        "t_center",
        "left_slope",
        "right_slope",
    ]

    return names


# ============================================================
# 5. ОБУЧЕНИЕ И ОЦЕНКА
# ============================================================

def train_scaled_target_regressor(model, X_train_features, y_train):
    """
    Масштабируем target-параметры.
    Это важно, потому что v_top/v_bottom измеряются тысячами,
    z0 — сотнями, а k — сотыми.
    """
    y_scaler = StandardScaler()
    y_train_scaled = y_scaler.fit_transform(y_train)
    model.fit(X_train_features, y_train_scaled)

    return model, y_scaler


def predict_scaled_target(model, y_scaler, X_features):
    y_pred_scaled = model.predict(X_features)
    return y_scaler.inverse_transform(y_pred_scaled)


def clip_params_to_bounds(params):
    clipped = np.array(params, dtype=float).copy()

    if clipped.ndim == 1:
        clipped = clipped.reshape(1, -1)

    for i, name in enumerate(PARAM_NAMES):
        low, high = PARAM_BOUNDS[name]
        clipped[:, i] = np.clip(clipped[:, i], low, high)

    return clipped


def evaluate_parameter_predictions(y_true, y_pred, model_name, test_noise_level):
    y_pred = clip_params_to_bounds(y_pred)

    rows = []

    for i, name in enumerate(PARAM_NAMES):
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse = math.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        r2 = r2_score(y_true[:, i], y_pred[:, i])
        normalized_mae = mae / PARAM_RANGES[i]

        rows.append({
            "model": model_name,
            "test_noise_level": test_noise_level,
            "parameter": name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "normalized_MAE": normalized_mae,
        })

    return pd.DataFrame(rows)


def evaluate_curve_rmse_for_subset(y_pred, X_true_curves, max_count):
    n = min(max_count, len(y_pred))
    rows = []

    for i in range(n):
        true_curve = X_true_curves[i]
        predicted_curve = simulate_curve(y_pred[i])
        rmse_ms = math.sqrt(mean_squared_error(true_curve, predicted_curve)) * 1000.0

        rows.append({
            "sample_index": i,
            "curve_RMSE_ms": rmse_ms,
        })

    return pd.DataFrame(rows)


# ============================================================
# 6. ЛОКАЛЬНОЕ УТОЧНЕНИЕ
# ============================================================

def curve_rmse_loss_ms(params, observed_curve):
    params = clip_params_to_bounds(np.array(params, dtype=float))[0]
    predicted_curve = simulate_curve(params)
    return math.sqrt(mean_squared_error(observed_curve, predicted_curve)) * 1000.0


def coordinate_refinement(initial_params, observed_curve, n_rounds=10):
    params = clip_params_to_bounds(np.array(initial_params, dtype=float))[0]

    steps = np.array([60.0, 180.0, 18.0, 0.004])
    best_loss = curve_rmse_loss_ms(params, observed_curve)

    history = []

    for round_idx in range(n_rounds):
        improved = False

        for j in range(len(params)):
            for direction in [-1, 1]:
                candidate = params.copy()
                candidate[j] += direction * steps[j]
                candidate = clip_params_to_bounds(candidate)[0]
                candidate_loss = curve_rmse_loss_ms(candidate, observed_curve)

                if candidate_loss < best_loss:
                    params = candidate
                    best_loss = candidate_loss
                    improved = True

        history.append({
            "round": round_idx,
            "loss_ms": best_loss,
            "v_top": params[0],
            "v_bottom": params[1],
            "z0": params[2],
            "k": params[3],
        })

        if improved:
            steps *= 0.80
        else:
            steps *= 0.55

    return params, best_loss, pd.DataFrame(history)


def multistart_refinement(initial_params, observed_curve, seed, n_starts=8):
    """
    Запускает локальное уточнение из нескольких начальных точек около прогноза регрессора.
    Это надежнее, чем один локальный запуск.
    """
    rng = np.random.default_rng(seed)

    starts = [np.array(initial_params, dtype=float)]

    perturb_scales = np.array([80.0, 220.0, 25.0, 0.006])
    for _ in range(n_starts - 1):
        candidate = np.array(initial_params, dtype=float) + rng.normal(0.0, perturb_scales)
        candidate = clip_params_to_bounds(candidate)[0]
        starts.append(candidate)

    best_params = None
    best_loss = np.inf
    best_history = None

    for start in starts:
        params, loss, history = coordinate_refinement(start, observed_curve, n_rounds=12)

        if loss < best_loss:
            best_params = params
            best_loss = loss
            best_history = history

    return best_params, best_loss, best_history


# ============================================================
# 7. ГРАФИКИ
# ============================================================

def plot_profile_comparison(true_params, reg_params, refined_params, filename):
    true_profile = profile_values_from_params(true_params)
    reg_profile = profile_values_from_params(reg_params)
    refined_profile = profile_values_from_params(refined_params)

    plt.figure(figsize=(6, 6))
    plt.plot(true_profile, ZS, label="true profile")
    plt.plot(reg_profile, ZS, "--", label="regressor estimate")
    plt.plot(refined_profile, ZS, ":", label="refined estimate")
    plt.gca().invert_yaxis()
    plt.xlabel("Скорость, м/с")
    plt.ylabel("Глубина z, м")
    plt.title("True vs regressor/refined velocity profile")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print("Сохранен график:", filename)


def plot_curve_comparison(true_curve, observed_curve, reg_curve, refined_curve, filename):
    plt.figure(figsize=(8, 5))
    plt.plot(RECEIVER_XS, true_curve, marker="o", label="clean true curve")
    plt.plot(RECEIVER_XS, observed_curve, marker="o", label="observed noisy curve")
    plt.plot(RECEIVER_XS, reg_curve, marker="o", linestyle="--", label="regressor curve")
    plt.plot(RECEIVER_XS, refined_curve, marker="o", linestyle=":", label="refined curve")
    plt.xlabel("x приемника, м")
    plt.ylabel("Время прихода, с")
    plt.title("Travel-time curve comparison")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print("Сохранен график:", filename)


def plot_refinement_history(history_df, filename):
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["round"], history_df["loss_ms"], marker="o")
    plt.xlabel("Round")
    plt.ylabel("RMSE, мс")
    plt.title("Multi-start local refinement loss history")
    plt.grid(True, alpha=0.3)
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print("Сохранен график:", filename)


def plot_feature_importance(model, feature_names, filename, top_n=25):
    if not hasattr(model, "feature_importances_"):
        return

    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:top_n]

    plt.figure(figsize=(10, 5))
    plt.bar(range(len(indices)), importances[indices])
    plt.xticks(range(len(indices)), [feature_names[i] for i in indices], rotation=60, ha="right")
    plt.ylabel("Importance")
    plt.title("Top feature importances")
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print("Сохранен график:", filename)


# ============================================================
# 8. ЗАПУСК ЭТАПА 9 CHECKED
# ============================================================

print("=" * 80)
print("ЭТАП 9 CHECKED: РЕГРЕССОР ДЛЯ ВОССТАНОВЛЕНИЯ ПАРАМЕТРОВ")
print("=" * 80)

print("RUN_MODE:", RUN_MODE)
print("N_BASE_SAMPLES:", N_BASE_SAMPLES)
print("N_ESTIMATORS:", N_ESTIMATORS)
print("TRAIN_NOISE_LEVELS:", TRAIN_NOISE_LEVELS)
print("TEST_NOISE_LEVELS:", TEST_NOISE_LEVELS)
print()

print("Генерируем датасет сигмоидальных профилей...")
X_curves, y_params = generate_regression_dataset(N_BASE_SAMPLES, RANDOM_SEED)

print()
print("Размер X_curves:", X_curves.shape)
print("Размер y_params:", y_params.shape)

dataset_df = pd.DataFrame(X_curves, columns=[f"t{i+1}" for i in range(X_curves.shape[1])])
for i, name in enumerate(PARAM_NAMES):
    dataset_df[name] = y_params[:, i]
dataset_df.to_csv("stage9_checked_sigmoid_regression_dataset.csv", index=False)

X_train, X_test, y_train, y_test = train_test_split(
    X_curves,
    y_params,
    test_size=0.25,
    random_state=RANDOM_SEED,
)

print()
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

X_train_aug, y_train_aug = make_noisy_training_copies(
    X_train,
    y_train,
    noise_levels=TRAIN_NOISE_LEVELS,
    seed=RANDOM_SEED + 500
)

print("Augmented train size:", X_train_aug.shape)

X_train_features = build_regression_features(X_train_aug)
feature_names = make_feature_names(X_curves.shape[1])

base_models = {
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
    "ExtraTreesRegressor": ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
}

trained_models = {}
all_parameter_summaries = []
overall_rows = []
curve_rmse_rows = []

for model_name, model in base_models.items():
    print()
    print("=" * 80)
    print("Обучаем:", model_name)
    print("=" * 80)

    fitted_model, y_scaler = train_scaled_target_regressor(
        model,
        X_train_features,
        y_train_aug
    )

    trained_models[model_name] = (fitted_model, y_scaler)

    plot_feature_importance(
        fitted_model,
        feature_names=feature_names,
        filename=f"stage9_checked_feature_importance_{model_name}.png",
        top_n=25
    )

    for noise_level in TEST_NOISE_LEVELS:
        print(f"Тестовый шум: {noise_level * 100:.1f}%")

        X_test_input = add_relative_noise(
            X_test,
            noise_level=noise_level,
            seed=RANDOM_SEED + int(noise_level * 100000) + 900
        )

        X_test_features = build_regression_features(X_test_input)

        y_pred = predict_scaled_target(fitted_model, y_scaler, X_test_features)
        y_pred = clip_params_to_bounds(y_pred)

        param_summary = evaluate_parameter_predictions(
            y_true=y_test,
            y_pred=y_pred,
            model_name=model_name,
            test_noise_level=noise_level
        )

        all_parameter_summaries.append(param_summary)

        overall_rows.append({
            "model": model_name,
            "test_noise_level": noise_level,
            "mean_normalized_MAE": param_summary["normalized_MAE"].mean(),
            "mean_R2": param_summary["R2"].mean(),
        })

        curve_rmse = evaluate_curve_rmse_for_subset(
            y_pred=y_pred,
            X_true_curves=X_test,
            max_count=MAX_CURVE_RMSE_TEST
        )

        curve_rmse["model"] = model_name
        curve_rmse["test_noise_level"] = noise_level

        curve_rmse_rows.append(curve_rmse)

        print("Parameter summary:")
        display(param_summary)

        print("Curve RMSE summary, ms:")
        display(curve_rmse["curve_RMSE_ms"].describe())

parameter_summary_df = pd.concat(all_parameter_summaries, ignore_index=True)
overall_df = pd.DataFrame(overall_rows).sort_values(
    ["mean_normalized_MAE", "test_noise_level"],
    ascending=[True, True]
)
curve_rmse_df = pd.concat(curve_rmse_rows, ignore_index=True)

parameter_summary_df.to_csv("stage9_checked_parameter_errors.csv", index=False)
overall_df.to_csv("stage9_checked_regressor_overall_comparison.csv", index=False)
curve_rmse_df.to_csv("stage9_checked_curve_rmse_all_models.csv", index=False)

curve_rmse_summary = (
    curve_rmse_df
    .groupby(["model", "test_noise_level"], as_index=False)
    .agg(
        mean_curve_RMSE_ms=("curve_RMSE_ms", "mean"),
        median_curve_RMSE_ms=("curve_RMSE_ms", "median"),
        max_curve_RMSE_ms=("curve_RMSE_ms", "max"),
    )
    .sort_values(["mean_curve_RMSE_ms", "test_noise_level"])
)
curve_rmse_summary.to_csv("stage9_checked_curve_rmse_summary.csv", index=False)

print()
print("=" * 80)
print("ИТОГОВОЕ СРАВНЕНИЕ РЕГРЕССОРОВ")
print("=" * 80)
display(overall_df)
display(curve_rmse_summary)
display(parameter_summary_df)

# Выбор лучшей модели по среднему normalized MAE по всем шумам.
overall_by_model = (
    overall_df
    .groupby("model", as_index=False)
    .agg(
        mean_normalized_MAE_over_noise=("mean_normalized_MAE", "mean"),
        min_mean_R2_over_noise=("mean_R2", "min"),
    )
    .sort_values("mean_normalized_MAE_over_noise")
)

overall_by_model.to_csv("stage9_checked_overall_by_model.csv", index=False)

print()
print("Сводка по моделям:")
display(overall_by_model)

best_model_name = overall_by_model.iloc[0]["model"]
print("Лучшая модель для демо:", best_model_name)


# ============================================================
# 9. DEMO: РЕГРЕССОР + MULTI-START REFINEMENT
# ============================================================

demo_index = 0

true_params = y_test[demo_index]
true_curve = X_test[demo_index]

observed_curve = add_relative_noise(
    true_curve.reshape(1, -1),
    DEMO_OBS_NOISE,
    RANDOM_SEED + 12345
)[0]

best_model, best_scaler = trained_models[best_model_name]
observed_features = build_regression_features(observed_curve.reshape(1, -1))

reg_params = predict_scaled_target(best_model, best_scaler, observed_features)
reg_params = clip_params_to_bounds(reg_params)[0]

refined_params, refined_loss_ms, refinement_history = multistart_refinement(
    initial_params=reg_params,
    observed_curve=observed_curve,
    seed=RANDOM_SEED + 777,
    n_starts=8
)

reg_curve = simulate_curve(reg_params)
refined_curve = simulate_curve(refined_params)

reg_loss_ms = math.sqrt(mean_squared_error(observed_curve, reg_curve)) * 1000.0
true_observed_noise_ms = math.sqrt(mean_squared_error(observed_curve, true_curve)) * 1000.0

demo_rows = []

for i, name in enumerate(PARAM_NAMES):
    demo_rows.append({
        "parameter": name,
        "true": true_params[i],
        "regressor_estimate": reg_params[i],
        "refined_estimate": refined_params[i],
        "regressor_abs_error": abs(reg_params[i] - true_params[i]),
        "refined_abs_error": abs(refined_params[i] - true_params[i]),
    })

demo_summary = pd.DataFrame(demo_rows)
demo_summary.to_csv("stage9_checked_demo_parameter_recovery_summary.csv", index=False)
refinement_history.to_csv("stage9_checked_demo_refinement_history.csv", index=False)

curve_summary = pd.DataFrame([
    {
        "metric": "observed_vs_true_clean_RMSE_ms",
        "value": true_observed_noise_ms,
    },
    {
        "metric": "observed_vs_regressor_RMSE_ms",
        "value": reg_loss_ms,
    },
    {
        "metric": "observed_vs_refined_RMSE_ms",
        "value": refined_loss_ms,
    },
])
curve_summary.to_csv("stage9_checked_demo_curve_rmse_summary.csv", index=False)

print()
print("=" * 80)
print("DEMO-ПРИМЕР: TRUE vs REGRESSOR vs REFINED")
print("=" * 80)
display(demo_summary)
display(curve_summary)

plot_profile_comparison(
    true_params=true_params,
    reg_params=reg_params,
    refined_params=refined_params,
    filename="stage9_checked_demo_profile_comparison.png"
)

plot_curve_comparison(
    true_curve=true_curve,
    observed_curve=observed_curve,
    reg_curve=reg_curve,
    refined_curve=refined_curve,
    filename="stage9_checked_demo_curve_comparison.png"
)

plot_refinement_history(
    refinement_history,
    filename="stage9_checked_demo_refinement_history.png"
)

print()
print("Готово. Основные файлы:")
print("stage9_checked_sigmoid_regression_dataset.csv")
print("stage9_checked_parameter_errors.csv")
print("stage9_checked_regressor_overall_comparison.csv")
print("stage9_checked_curve_rmse_all_models.csv")
print("stage9_checked_curve_rmse_summary.csv")
print("stage9_checked_overall_by_model.csv")
print("stage9_checked_feature_importance_RandomForestRegressor.png")
print("stage9_checked_feature_importance_ExtraTreesRegressor.png")
print("stage9_checked_demo_parameter_recovery_summary.csv")
print("stage9_checked_demo_curve_rmse_summary.csv")
print("stage9_checked_demo_profile_comparison.png")
print("stage9_checked_demo_curve_comparison.png")
print("stage9_checked_demo_refinement_history.png")


# ============================================================
# 10. СКАЧИВАНИЕ РЕЗУЛЬТАТОВ
# ============================================================

def download_stage9_checked_results():
    import zipfile
    from pathlib import Path
    from google.colab import files

    base_dir = Path("/content")
    zip_path = base_dir / "stage9_checked_regression_inverse_results.zip"

    result_files = []
    for path in base_dir.rglob("*"):
        if path.is_file():
            if path.name.startswith("stage9_checked_") and (path.name.endswith(".png") or path.name.endswith(".csv")):
                result_files.append(path)

    result_files = sorted(set(result_files))

    print("Найдено файлов:", len(result_files))
    for path in result_files:
        print(path.name)

    if len(result_files) == 0:
        print("Файлы stage9_checked не найдены.")
        return

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for path in result_files:
            zipf.write(path, arcname=path.name)

    print("Архив создан:", zip_path)
    files.download(str(zip_path))


print()
print("Чтобы скачать все результаты, выполни в новой ячейке:")
print("download_stage9_checked_results()")

## Этап 9.1 — мульти-источниковая регрессия

Этап 9.1 CHECKED:
мульти-источниковая обучаемая обратная модель.

Зачем нужен этап:
На этапе 9 одна travel-time curve от одного источника давала ограниченную информацию.
Из-за этого разные параметры сигмоидального профиля могли давать похожие кривые,
особенно плохо восстанавливался параметр k — резкость перехода.

Главная идея этапа 9.1:
использовать не одну, а несколько travel-time curves от разных глубин источника.

    source_z = 250 м
    source_z = 350 м
    source_z = 450 м

Для одного и того же профиля скорости считаются три кривые времен прихода.
Они "просвечивают" среду по-разному и должны дать регрессору больше информации
о параметрах профиля.

Что восстанавливаем:
сигмоидальный профиль скорости

    v(z) = v_top + (v_bottom - v_top) / (1 + exp(-k * (z - z0)))

Параметры:
- v_top    — скорость в верхней части среды;
- v_bottom — скорость в нижней части среды;
- z0       — глубина перехода;
- k        — резкость перехода.

Что проверяет ноутбук:
1. Генерирует датасет параметров сигмоидальных профилей.
2. Для каждого профиля считает несколько travel-time curves от разных источников.
3. Сравнивает single-source и multi-source режимы на одном и том же датасете.
4. Обучает RandomForestRegressor и ExtraTreesRegressor.
5. Проверяет устойчивость на тестовом шуме 0%, 0.5%, 1%, 2%, 5%.
6. Считает:
   - ошибки параметров;
   - curve RMSE в миллисекундах;
   - сводную таблицу по моделям.
7. Для demo-примера делает:
   - прогноз регрессора;
   - multi-start local refinement;
   - сравнение true/regressor/refined профиля;
   - сравнение travel-time curves для каждого источника.

Методологически важно:
- сначала создается clean dataset;
- затем делается train/test split;
- шумовые копии создаются только для train;
- test не попадает в augmentation;
- тестовый шум добавляется отдельно и независимо.

In [ ]:
import heapq
import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

RUN_MODE = "FAST"  # "FAST" или "ACCURATE"

if RUN_MODE == "ACCURATE":
    N_BASE_SAMPLES = 1200
    N_ESTIMATORS = 300
    MAX_CURVE_RMSE_TEST = 60
else:
    N_BASE_SAMPLES = 4000
    N_ESTIMATORS = 600
    MAX_CURVE_RMSE_TEST = 150

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Несколько источников.
SOURCE_Z_LIST = [250.0, 350.0, 450.0]

# Для single-source baseline берем центральный по глубине источник.
SINGLE_SOURCE_INDEX = 1

TRAIN_NOISE_LEVELS = [0.0, 0.005, 0.01, 0.02, 0.05]
TEST_NOISE_LEVELS = [0.0, 0.005, 0.01, 0.02, 0.05]

DEMO_OBS_NOISE = 0.01


@dataclass
class ModelConfig:
    x_min: float = 0.0
    x_max: float = 1200.0
    z_min: float = 0.0
    z_max: float = 500.0

    # Сетка не слишком тяжелая, потому что прямую задачу надо решать много раз.
    nx: int = 51
    nz: int = 41

    source_x: float = 600.0

    receiver_count: int = 21
    receiver_z: float = 0.0


CONFIG = ModelConfig()

PARAM_NAMES = ["v_top", "v_bottom", "z0", "k"]

PARAM_BOUNDS = {
    "v_top": (1200.0, 2600.0),
    "v_bottom": (3000.0, 7000.0),
    "z0": (80.0, 420.0),
    "k": (0.005, 0.080),
}

PARAM_RANGES = np.array([
    PARAM_BOUNDS["v_top"][1] - PARAM_BOUNDS["v_top"][0],
    PARAM_BOUNDS["v_bottom"][1] - PARAM_BOUNDS["v_bottom"][0],
    PARAM_BOUNDS["z0"][1] - PARAM_BOUNDS["z0"][0],
    PARAM_BOUNDS["k"][1] - PARAM_BOUNDS["k"][0],
])


# ============================================================
# 2. СЕТКА И ПРЯМАЯ МОДЕЛЬ
# ============================================================

def build_grid(config):
    xs = np.linspace(config.x_min, config.x_max, config.nx)
    zs = np.linspace(config.z_min, config.z_max, config.nz)
    dx = xs[1] - xs[0]
    dz = zs[1] - zs[0]
    return xs, zs, dx, dz


XS, ZS, DX, DZ = build_grid(CONFIG)

SOURCE_IX = int(np.argmin(np.abs(XS - CONFIG.source_x)))

SOURCE_IZ_LIST = [
    int(np.argmin(np.abs(ZS - source_z)))
    for source_z in SOURCE_Z_LIST
]

RECEIVER_XS = np.linspace(
    CONFIG.x_min + 50.0,
    CONFIG.x_max - 50.0,
    CONFIG.receiver_count
)

RECEIVER_IZ = int(np.argmin(np.abs(ZS - CONFIG.receiver_z)))
RECEIVER_IXS = [int(np.argmin(np.abs(XS - x))) for x in RECEIVER_XS]


def sigmoid_profile(v_top, v_bottom, z0, k):
    def profile(z):
        return v_top + (v_bottom - v_top) / (1.0 + math.exp(-k * (z - z0)))
    return profile


def profile_values_from_params(params):
    v_top, v_bottom, z0, k = params
    profile = sigmoid_profile(v_top, v_bottom, z0, k)
    return np.array([profile(z) for z in ZS])


def build_velocity_model_from_params(params):
    velocity_model = np.zeros((len(ZS), len(XS)))
    profile_values = profile_values_from_params(params)

    for iz, velocity in enumerate(profile_values):
        velocity_model[iz, :] = velocity

    return velocity_model


def dijkstra_travel_times(velocity_model, source_iz):
    """
    Прямая задача для одного источника:
    находим минимальные времена прихода от source к receivers.
    """
    nz, nx = velocity_model.shape

    travel_times = np.full((nz, nx), np.inf)
    travel_times[source_iz, SOURCE_IX] = 0.0

    neighbors = [
        (-1, 0), (1, 0), (0, -1), (0, 1),
        (-1, -1), (-1, 1), (1, -1), (1, 1),
    ]

    heap = [(0.0, source_iz, SOURCE_IX)]

    while heap:
        current_time, iz, ix = heapq.heappop(heap)

        if current_time > travel_times[iz, ix]:
            continue

        for dzi, dxi in neighbors:
            next_iz = iz + dzi
            next_ix = ix + dxi

            if not (0 <= next_iz < nz and 0 <= next_ix < nx):
                continue

            step_distance = math.sqrt((dxi * DX) ** 2 + (dzi * DZ) ** 2)
            average_velocity = 0.5 * (
                velocity_model[iz, ix] + velocity_model[next_iz, next_ix]
            )

            step_time = step_distance / average_velocity
            new_time = current_time + step_time

            if new_time < travel_times[next_iz, next_ix]:
                travel_times[next_iz, next_ix] = new_time
                heapq.heappush(heap, (new_time, next_iz, next_ix))

    receiver_times = np.array([
        travel_times[RECEIVER_IZ, receiver_ix]
        for receiver_ix in RECEIVER_IXS
    ])

    return receiver_times


def simulate_multisource_curves(params):
    """
    Возвращает matrix shape:
        (n_sources, n_receivers)
    """
    velocity_model = build_velocity_model_from_params(params)

    curves = []
    for source_iz in SOURCE_IZ_LIST:
        curve = dijkstra_travel_times(velocity_model, source_iz)
        curves.append(curve)

    return np.array(curves)


def simulate_multisource_flat(params):
    return simulate_multisource_curves(params).reshape(-1)


# ============================================================
# 3. ДАТАСЕТ
# ============================================================

def sample_sigmoid_params(rng):
    v_top = rng.uniform(*PARAM_BOUNDS["v_top"])
    v_bottom = rng.uniform(*PARAM_BOUNDS["v_bottom"])
    z0 = rng.uniform(*PARAM_BOUNDS["z0"])
    k = rng.uniform(*PARAM_BOUNDS["k"])

    return np.array([v_top, v_bottom, z0, k], dtype=float)


def generate_multisource_dataset(n_samples, seed):
    rng = np.random.default_rng(seed)

    X_flat = []
    y_params = []

    for i in range(n_samples):
        params = sample_sigmoid_params(rng)
        curves = simulate_multisource_curves(params)

        X_flat.append(curves.reshape(-1))
        y_params.append(params)

        if (i + 1) % 50 == 0:
            print(f"Сгенерировано {i + 1}/{n_samples} примеров...")

    return np.array(X_flat), np.array(y_params)


def add_relative_noise(X, noise_level, seed):
    if noise_level == 0:
        return X.copy()

    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, noise_level * X, size=X.shape)

    return np.maximum(X + noise, 1e-6)


def make_noisy_training_copies(X_train, y_train, noise_levels, seed):
    X_parts = []
    y_parts = []

    for i, noise_level in enumerate(noise_levels):
        X_noisy = add_relative_noise(
            X_train,
            noise_level=noise_level,
            seed=seed + i * 1000 + int(noise_level * 10000)
        )
        X_parts.append(X_noisy)
        y_parts.append(y_train.copy())

    return np.vstack(X_parts), np.vstack(y_parts)


# ============================================================
# 4. ПРИЗНАКИ
# ============================================================

def reshape_flat_to_sources(X_flat):
    X_flat = np.asarray(X_flat)
    return X_flat.reshape(X_flat.shape[0], len(SOURCE_Z_LIST), CONFIG.receiver_count)


def select_sources(X_flat, input_mode):
    X_sources = reshape_flat_to_sources(X_flat)

    if input_mode == "single_source":
        return X_sources[:, [SINGLE_SOURCE_INDEX], :]

    if input_mode == "multi_source":
        return X_sources

    raise ValueError("Unknown input_mode")


def build_features_from_sources(X_selected):
    """
    X_selected shape:
        (n_samples, n_selected_sources, n_receivers)

    Признаки:
    - raw times для каждого источника;
    - normalized shape для каждого источника;
    - d1 для каждого источника;
    - summary features для каждого источника;
    - если источников несколько: разности между соседними source curves.
    """
    n_samples, n_sources, n_receivers = X_selected.shape

    feature_parts = []

    for s in range(n_sources):
        X = X_selected[:, s, :]

        raw = X

        x_min = X.min(axis=1, keepdims=True)
        x_range = X.max(axis=1, keepdims=True) - X.min(axis=1, keepdims=True)
        x_range = np.maximum(x_range, 1e-9)
        normalized = (X - x_min) / x_range

        d1 = np.gradient(X, axis=1)

        center_idx = n_receivers // 2
        left_slope = X[:, center_idx] - X[:, 0]
        right_slope = X[:, -1] - X[:, center_idx]

        summary = np.column_stack([
            X.min(axis=1),
            X.max(axis=1),
            X.mean(axis=1),
            X.std(axis=1),
            X.max(axis=1) - X.min(axis=1),
            X[:, center_idx],
            left_slope,
            right_slope,
        ])

        feature_parts += [raw, normalized, d1, summary]

    if n_sources >= 2:
        # Разности кривых от разных глубин источника.
        # Это должно помочь регрессору видеть, как меняется ответ среды
        # при изменении глубины источника.
        for s in range(n_sources - 1):
            diff = X_selected[:, s + 1, :] - X_selected[:, s, :]
            feature_parts.append(diff)

            diff_summary = np.column_stack([
                diff.min(axis=1),
                diff.max(axis=1),
                diff.mean(axis=1),
                diff.std(axis=1),
                diff.max(axis=1) - diff.min(axis=1),
            ])
            feature_parts.append(diff_summary)

    return np.hstack(feature_parts)


def build_regression_features(X_flat, input_mode):
    X_selected = select_sources(X_flat, input_mode)
    return build_features_from_sources(X_selected)


def make_feature_names(input_mode):
    if input_mode == "single_source":
        source_labels = [f"src{SINGLE_SOURCE_INDEX}_z{int(SOURCE_Z_LIST[SINGLE_SOURCE_INDEX])}"]
    else:
        source_labels = [f"src{s}_z{int(z)}" for s, z in enumerate(SOURCE_Z_LIST)]

    names = []

    for source_label in source_labels:
        for i in range(CONFIG.receiver_count):
            names.append(f"{source_label}_t{i+1}")

        for i in range(CONFIG.receiver_count):
            names.append(f"{source_label}_tnorm{i+1}")

        for i in range(CONFIG.receiver_count):
            names.append(f"{source_label}_d1_{i+1}")

        names += [
            f"{source_label}_t_min",
            f"{source_label}_t_max",
            f"{source_label}_t_mean",
            f"{source_label}_t_std",
            f"{source_label}_t_range",
            f"{source_label}_t_center",
            f"{source_label}_left_slope",
            f"{source_label}_right_slope",
        ]

    if input_mode == "multi_source":
        for s in range(len(SOURCE_Z_LIST) - 1):
            label = f"diff_src{s+1}_minus_src{s}"
            for i in range(CONFIG.receiver_count):
                names.append(f"{label}_{i+1}")

            names += [
                f"{label}_min",
                f"{label}_max",
                f"{label}_mean",
                f"{label}_std",
                f"{label}_range",
            ]

    return names


# ============================================================
# 5. ОБУЧЕНИЕ И ОЦЕНКА
# ============================================================

def train_scaled_target_regressor(model, X_train_features, y_train):
    y_scaler = StandardScaler()
    y_train_scaled = y_scaler.fit_transform(y_train)
    model.fit(X_train_features, y_train_scaled)

    return model, y_scaler


def predict_scaled_target(model, y_scaler, X_features):
    y_pred_scaled = model.predict(X_features)
    return y_scaler.inverse_transform(y_pred_scaled)


def clip_params_to_bounds(params):
    clipped = np.array(params, dtype=float).copy()

    if clipped.ndim == 1:
        clipped = clipped.reshape(1, -1)

    for i, name in enumerate(PARAM_NAMES):
        low, high = PARAM_BOUNDS[name]
        clipped[:, i] = np.clip(clipped[:, i], low, high)

    return clipped


def evaluate_parameter_predictions(y_true, y_pred, model_name, input_mode, test_noise_level):
    y_pred = clip_params_to_bounds(y_pred)

    rows = []

    for i, name in enumerate(PARAM_NAMES):
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse = math.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        r2 = r2_score(y_true[:, i], y_pred[:, i])
        normalized_mae = mae / PARAM_RANGES[i]

        rows.append({
            "model": model_name,
            "input_mode": input_mode,
            "test_noise_level": test_noise_level,
            "parameter": name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "normalized_MAE": normalized_mae,
        })

    return pd.DataFrame(rows)


def evaluate_curve_rmse_for_subset(y_pred, X_true_flat, max_count):
    n = min(max_count, len(y_pred))
    rows = []

    for i in range(n):
        true_flat = X_true_flat[i]
        predicted_flat = simulate_multisource_flat(y_pred[i])
        rmse_ms = math.sqrt(mean_squared_error(true_flat, predicted_flat)) * 1000.0

        rows.append({
            "sample_index": i,
            "multisource_curve_RMSE_ms": rmse_ms,
        })

    return pd.DataFrame(rows)


# ============================================================
# 6. ЛОКАЛЬНОЕ УТОЧНЕНИЕ
# ============================================================

def curve_rmse_loss_ms(params, observed_flat):
    params = clip_params_to_bounds(np.array(params, dtype=float))[0]
    predicted_flat = simulate_multisource_flat(params)
    return math.sqrt(mean_squared_error(observed_flat, predicted_flat)) * 1000.0


def coordinate_refinement(initial_params, observed_flat, n_rounds=10):
    params = clip_params_to_bounds(np.array(initial_params, dtype=float))[0]

    steps = np.array([60.0, 180.0, 18.0, 0.004])
    best_loss = curve_rmse_loss_ms(params, observed_flat)

    history = []

    for round_idx in range(n_rounds):
        improved = False

        for j in range(len(params)):
            for direction in [-1, 1]:
                candidate = params.copy()
                candidate[j] += direction * steps[j]
                candidate = clip_params_to_bounds(candidate)[0]

                candidate_loss = curve_rmse_loss_ms(candidate, observed_flat)

                if candidate_loss < best_loss:
                    params = candidate
                    best_loss = candidate_loss
                    improved = True

        history.append({
            "round": round_idx,
            "loss_ms": best_loss,
            "v_top": params[0],
            "v_bottom": params[1],
            "z0": params[2],
            "k": params[3],
        })

        if improved:
            steps *= 0.80
        else:
            steps *= 0.55

    return params, best_loss, pd.DataFrame(history)


def multistart_refinement(initial_params, observed_flat, seed, n_starts=10):
    rng = np.random.default_rng(seed)

    starts = [np.array(initial_params, dtype=float)]

    perturb_scales = np.array([90.0, 250.0, 30.0, 0.007])

    for _ in range(n_starts - 1):
        candidate = np.array(initial_params, dtype=float) + rng.normal(0.0, perturb_scales)
        candidate = clip_params_to_bounds(candidate)[0]
        starts.append(candidate)

    best_params = None
    best_loss = np.inf
    best_history = None

    for start in starts:
        params, loss, history = coordinate_refinement(start, observed_flat, n_rounds=12)

        if loss < best_loss:
            best_params = params
            best_loss = loss
            best_history = history

    return best_params, best_loss, best_history


# ============================================================
# 7. ГРАФИКИ
# ============================================================

def plot_profile_comparison(true_params, reg_params, refined_params, filename):
    true_profile = profile_values_from_params(true_params)
    reg_profile = profile_values_from_params(reg_params)
    refined_profile = profile_values_from_params(refined_params)

    plt.figure(figsize=(6, 6))
    plt.plot(true_profile, ZS, label="true profile")
    plt.plot(reg_profile, ZS, "--", label="regressor estimate")
    plt.plot(refined_profile, ZS, ":", label="refined estimate")
    plt.gca().invert_yaxis()
    plt.xlabel("Скорость, м/с")
    plt.ylabel("Глубина z, м")
    plt.title("True vs regressor/refined velocity profile")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print("Сохранен график:", filename)


def plot_curve_comparison_by_source(true_flat, observed_flat, reg_flat, refined_flat, filename_prefix):
    true_curves = true_flat.reshape(len(SOURCE_Z_LIST), CONFIG.receiver_count)
    observed_curves = observed_flat.reshape(len(SOURCE_Z_LIST), CONFIG.receiver_count)
    reg_curves = reg_flat.reshape(len(SOURCE_Z_LIST), CONFIG.receiver_count)
    refined_curves = refined_flat.reshape(len(SOURCE_Z_LIST), CONFIG.receiver_count)

    for s, source_z in enumerate(SOURCE_Z_LIST):
        plt.figure(figsize=(8, 5))
        plt.plot(RECEIVER_XS, true_curves[s], marker="o", label="clean true curve")
        plt.plot(RECEIVER_XS, observed_curves[s], marker="o", label="observed noisy curve")
        plt.plot(RECEIVER_XS, reg_curves[s], marker="o", linestyle="--", label="regressor curve")
        plt.plot(RECEIVER_XS, refined_curves[s], marker="o", linestyle=":", label="refined curve")
        plt.xlabel("x приемника, м")
        plt.ylabel("Время прихода, с")
        plt.title(f"Travel-time curve comparison, source_z={source_z:.0f} m")
        plt.grid(True, alpha=0.3)
        plt.legend()
        filename = f"{filename_prefix}_source_z_{int(source_z)}.png"
        plt.savefig(filename, dpi=200, bbox_inches="tight")
        plt.show()
        print("Сохранен график:", filename)


def plot_refinement_history(history_df, filename):
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["round"], history_df["loss_ms"], marker="o")
    plt.xlabel("Round")
    plt.ylabel("RMSE, мс")
    plt.title("Multi-start local refinement loss history")
    plt.grid(True, alpha=0.3)
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print("Сохранен график:", filename)


def plot_feature_importance(model, feature_names, filename, top_n=25):
    if not hasattr(model, "feature_importances_"):
        return

    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:top_n]

    plt.figure(figsize=(10, 5))
    plt.bar(range(len(indices)), importances[indices])
    plt.xticks(range(len(indices)), [feature_names[i] for i in indices], rotation=60, ha="right")
    plt.ylabel("Importance")
    plt.title("Top feature importances")
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print("Сохранен график:", filename)


# ============================================================
# 8. ЗАПУСК ЭТАПА 9.1
# ============================================================

print("=" * 80)
print("ЭТАП 9.1 CHECKED: MULTI-SOURCE REGRESSION INVERSE MODEL")
print("=" * 80)

print("RUN_MODE:", RUN_MODE)
print("N_BASE_SAMPLES:", N_BASE_SAMPLES)
print("N_ESTIMATORS:", N_ESTIMATORS)
print("SOURCE_Z_LIST:", SOURCE_Z_LIST)
print("TRAIN_NOISE_LEVELS:", TRAIN_NOISE_LEVELS)
print("TEST_NOISE_LEVELS:", TEST_NOISE_LEVELS)
print()

print("Генерируем multi-source датасет...")
X_flat, y_params = generate_multisource_dataset(N_BASE_SAMPLES, RANDOM_SEED)

print()
print("Размер X_flat:", X_flat.shape)
print("Размер y_params:", y_params.shape)

dataset_df = pd.DataFrame(X_flat, columns=[
    f"source{s}_z{int(z)}_t{i+1}"
    for s, z in enumerate(SOURCE_Z_LIST)
    for i in range(CONFIG.receiver_count)
])

for i, name in enumerate(PARAM_NAMES):
    dataset_df[name] = y_params[:, i]

dataset_df.to_csv("stage9_1_checked_multisource_regression_dataset.csv", index=False)

X_train, X_test, y_train, y_test = train_test_split(
    X_flat,
    y_params,
    test_size=0.25,
    random_state=RANDOM_SEED,
)

print()
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

X_train_aug, y_train_aug = make_noisy_training_copies(
    X_train,
    y_train,
    noise_levels=TRAIN_NOISE_LEVELS,
    seed=RANDOM_SEED + 500
)

print("Augmented train size:", X_train_aug.shape)

base_models = {
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
    "ExtraTreesRegressor": ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
}

INPUT_MODES = ["single_source", "multi_source"]

trained_models = {}
all_parameter_summaries = []
overall_rows = []
curve_rmse_rows = []

for input_mode in INPUT_MODES:
    print()
    print("=" * 80)
    print("INPUT MODE:", input_mode)
    print("=" * 80)

    X_train_features = build_regression_features(X_train_aug, input_mode=input_mode)
    feature_names = make_feature_names(input_mode)

    for model_name, model_template in base_models.items():
        print()
        print("-" * 80)
        print("Обучаем:", model_name, "| input_mode:", input_mode)
        print("-" * 80)

        # Создаем новую модель, чтобы не переиспользовать уже обученный объект.
        if model_name == "RandomForestRegressor":
            model = RandomForestRegressor(
                n_estimators=N_ESTIMATORS,
                max_depth=None,
                min_samples_leaf=2,
                random_state=RANDOM_SEED,
                n_jobs=-1,
            )
        else:
            model = ExtraTreesRegressor(
                n_estimators=N_ESTIMATORS,
                max_depth=None,
                min_samples_leaf=2,
                random_state=RANDOM_SEED,
                n_jobs=-1,
            )

        fitted_model, y_scaler = train_scaled_target_regressor(
            model,
            X_train_features,
            y_train_aug
        )

        trained_models[(model_name, input_mode)] = (fitted_model, y_scaler)

        plot_feature_importance(
            fitted_model,
            feature_names=feature_names,
            filename=f"stage9_1_checked_feature_importance_{model_name}_{input_mode}.png",
            top_n=25
        )

        for noise_level in TEST_NOISE_LEVELS:
            print(f"Тестовый шум: {noise_level * 100:.1f}%")

            X_test_input = add_relative_noise(
                X_test,
                noise_level=noise_level,
                seed=RANDOM_SEED + int(noise_level * 100000) + 900
            )

            X_test_features = build_regression_features(X_test_input, input_mode=input_mode)

            y_pred = predict_scaled_target(fitted_model, y_scaler, X_test_features)
            y_pred = clip_params_to_bounds(y_pred)

            param_summary = evaluate_parameter_predictions(
                y_true=y_test,
                y_pred=y_pred,
                model_name=model_name,
                input_mode=input_mode,
                test_noise_level=noise_level
            )

            all_parameter_summaries.append(param_summary)

            overall_rows.append({
                "model": model_name,
                "input_mode": input_mode,
                "test_noise_level": noise_level,
                "mean_normalized_MAE": param_summary["normalized_MAE"].mean(),
                "mean_R2": param_summary["R2"].mean(),
            })

            curve_rmse = evaluate_curve_rmse_for_subset(
                y_pred=y_pred,
                X_true_flat=X_test,
                max_count=MAX_CURVE_RMSE_TEST
            )
            curve_rmse["model"] = model_name
            curve_rmse["input_mode"] = input_mode
            curve_rmse["test_noise_level"] = noise_level

            curve_rmse_rows.append(curve_rmse)

            print("Parameter summary:")
            display(param_summary)

            print("Multi-source curve RMSE summary, ms:")
            display(curve_rmse["multisource_curve_RMSE_ms"].describe())

parameter_summary_df = pd.concat(all_parameter_summaries, ignore_index=True)
overall_df = pd.DataFrame(overall_rows).sort_values(
    ["mean_normalized_MAE", "test_noise_level"],
    ascending=[True, True]
)
curve_rmse_df = pd.concat(curve_rmse_rows, ignore_index=True)

parameter_summary_df.to_csv("stage9_1_checked_parameter_errors.csv", index=False)
overall_df.to_csv("stage9_1_checked_regressor_overall_comparison.csv", index=False)
curve_rmse_df.to_csv("stage9_1_checked_curve_rmse_all_models.csv", index=False)

curve_rmse_summary = (
    curve_rmse_df
    .groupby(["model", "input_mode", "test_noise_level"], as_index=False)
    .agg(
        mean_curve_RMSE_ms=("multisource_curve_RMSE_ms", "mean"),
        median_curve_RMSE_ms=("multisource_curve_RMSE_ms", "median"),
        max_curve_RMSE_ms=("multisource_curve_RMSE_ms", "max"),
    )
    .sort_values(["mean_curve_RMSE_ms", "test_noise_level"])
)
curve_rmse_summary.to_csv("stage9_1_checked_curve_rmse_summary.csv", index=False)

overall_by_model = (
    overall_df
    .groupby(["model", "input_mode"], as_index=False)
    .agg(
        mean_normalized_MAE_over_noise=("mean_normalized_MAE", "mean"),
        min_mean_R2_over_noise=("mean_R2", "min"),
    )
    .sort_values("mean_normalized_MAE_over_noise")
)
overall_by_model.to_csv("stage9_1_checked_overall_by_model.csv", index=False)

print()
print("=" * 80)
print("ИТОГОВОЕ СРАВНЕНИЕ")
print("=" * 80)
display(overall_by_model)
display(overall_df)
display(curve_rmse_summary)
display(parameter_summary_df)

best_model_name = overall_by_model.iloc[0]["model"]
best_input_mode = overall_by_model.iloc[0]["input_mode"]

print()
print("Лучшая модель для demo:")
print("model:", best_model_name)
print("input_mode:", best_input_mode)


# ============================================================
# 9. DEMO
# ============================================================

demo_index = 0

true_params = y_test[demo_index]
true_flat = X_test[demo_index]

observed_flat = add_relative_noise(
    true_flat.reshape(1, -1),
    DEMO_OBS_NOISE,
    RANDOM_SEED + 12345
)[0]

best_model, best_scaler = trained_models[(best_model_name, best_input_mode)]

observed_features = build_regression_features(
    observed_flat.reshape(1, -1),
    input_mode=best_input_mode
)

reg_params = predict_scaled_target(best_model, best_scaler, observed_features)
reg_params = clip_params_to_bounds(reg_params)[0]

refined_params, refined_loss_ms, refinement_history = multistart_refinement(
    initial_params=reg_params,
    observed_flat=observed_flat,
    seed=RANDOM_SEED + 777,
    n_starts=10
)

reg_flat = simulate_multisource_flat(reg_params)
refined_flat = simulate_multisource_flat(refined_params)

reg_loss_ms = math.sqrt(mean_squared_error(observed_flat, reg_flat)) * 1000.0
true_observed_noise_ms = math.sqrt(mean_squared_error(observed_flat, true_flat)) * 1000.0

demo_rows = []

for i, name in enumerate(PARAM_NAMES):
    demo_rows.append({
        "parameter": name,
        "true": true_params[i],
        "regressor_estimate": reg_params[i],
        "refined_estimate": refined_params[i],
        "regressor_abs_error": abs(reg_params[i] - true_params[i]),
        "refined_abs_error": abs(refined_params[i] - true_params[i]),
    })

demo_summary = pd.DataFrame(demo_rows)
demo_summary.to_csv("stage9_1_checked_demo_parameter_recovery_summary.csv", index=False)
refinement_history.to_csv("stage9_1_checked_demo_refinement_history.csv", index=False)

curve_summary = pd.DataFrame([
    {
        "metric": "observed_vs_true_clean_multisource_RMSE_ms",
        "value": true_observed_noise_ms,
    },
    {
        "metric": "observed_vs_regressor_multisource_RMSE_ms",
        "value": reg_loss_ms,
    },
    {
        "metric": "observed_vs_refined_multisource_RMSE_ms",
        "value": refined_loss_ms,
    },
])
curve_summary.to_csv("stage9_1_checked_demo_curve_rmse_summary.csv", index=False)

print()
print("=" * 80)
print("DEMO-ПРИМЕР")
print("=" * 80)
print("best_model:", best_model_name)
print("best_input_mode:", best_input_mode)
display(demo_summary)
display(curve_summary)

plot_profile_comparison(
    true_params=true_params,
    reg_params=reg_params,
    refined_params=refined_params,
    filename="stage9_1_checked_demo_profile_comparison.png"
)

plot_curve_comparison_by_source(
    true_flat=true_flat,
    observed_flat=observed_flat,
    reg_flat=reg_flat,
    refined_flat=refined_flat,
    filename_prefix="stage9_1_checked_demo_curve_comparison"
)

plot_refinement_history(
    refinement_history,
    filename="stage9_1_checked_demo_refinement_history.png"
)

print()
print("Готово. Основные файлы:")
print("stage9_1_checked_multisource_regression_dataset.csv")
print("stage9_1_checked_parameter_errors.csv")
print("stage9_1_checked_regressor_overall_comparison.csv")
print("stage9_1_checked_curve_rmse_all_models.csv")
print("stage9_1_checked_curve_rmse_summary.csv")
print("stage9_1_checked_overall_by_model.csv")
print("stage9_1_checked_demo_parameter_recovery_summary.csv")
print("stage9_1_checked_demo_curve_rmse_summary.csv")
print("stage9_1_checked_demo_profile_comparison.png")
print("stage9_1_checked_demo_curve_comparison_source_z_250.png")
print("stage9_1_checked_demo_curve_comparison_source_z_350.png")
print("stage9_1_checked_demo_curve_comparison_source_z_450.png")
print("stage9_1_checked_demo_refinement_history.png")


# ============================================================
# 10. СКАЧАТЬ РЕЗУЛЬТАТЫ
# ============================================================

def download_stage9_1_checked_results():
    import zipfile
    from pathlib import Path
    from google.colab import files

    base_dir = Path("/content")
    zip_path = base_dir / "stage9_1_checked_multisource_regression_results.zip"

    result_files = []
    for path in base_dir.rglob("*"):
        if path.is_file():
            if path.name.startswith("stage9_1_checked_") and (path.name.endswith(".png") or path.name.endswith(".csv")):
                result_files.append(path)

    result_files = sorted(set(result_files))

    print("Найдено файлов:", len(result_files))
    for path in result_files:
        print(path.name)

    if len(result_files) == 0:
        print("Файлы stage9_1_checked не найдены.")
        return

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for path in result_files:
            zipf.write(path, arcname=path.name)

    print("Архив создан:", zip_path)
    files.download(str(zip_path))


print()
print("Чтобы скачать все результаты, выполни в новой ячейке:")
print("download_stage9_1_checked_results()")

## Этап 10 — гибридная инверсия

In [ ]:

# ЭТАП 10 CHECKED v2: гибридная multi-source инверсия
# Сравнение методов:
# 1) random_search_baseline
# 2) ExtraTrees_only
# 3) ExtraTrees_plus_local
# 4) ExtraTrees_plus_local_plus_MCMC_MAP
#
# v2 исправляет MCMC-адаптацию proposal и уменьшает лишние расчеты прямой задачи.
#
# Смысл этапа:
# не просто "добавить MCMC", а проверить, что реально улучшает восстановление
# параметров профиля и что только лучше подгоняет travel-time curves.

import heapq
import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from IPython.display import display
except Exception:
    display = print

from pathlib import Path
import os

RESULTS_DIR = Path("/content/stage10_accurate_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(RESULTS_DIR)

print("Все результаты будут сохраняться сюда:")
print(Path.cwd())



# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

RUN_MODE = "ACCURATE"  # "FAST" или "ACCURATE"

if RUN_MODE == "FAST":
    N_BASE_SAMPLES = 1200
    N_ESTIMATORS = 300
    BENCHMARK_SAMPLES = 6
    RANDOM_SEARCH_TRIALS = 50
    LOCAL_STARTS = 5
    MCMC_STEPS = 650
    MCMC_BURN_IN = 180
    MCMC_THIN = 2
elif RUN_MODE == "ACCURATE":
    N_BASE_SAMPLES = 4000
    N_ESTIMATORS = 600
    BENCHMARK_SAMPLES = 20
    RANDOM_SEARCH_TRIALS = 100
    LOCAL_STARTS = 8
    MCMC_STEPS = 1200
    MCMC_BURN_IN = 300
    MCMC_THIN = 3
else:
    raise ValueError("RUN_MODE должен быть FAST или ACCURATE")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

SOURCE_Z_LIST = [250.0, 350.0, 450.0]
TRAIN_NOISE_LEVELS = [0.0, 0.005, 0.01, 0.02, 0.05]
FULL_TEST_NOISE_LEVELS = [0.0, 0.005, 0.01, 0.02, 0.05]
BENCHMARK_NOISE_LEVELS = [0.0, 0.01, 0.05]
DEMO_NOISE_LEVEL = 0.01

PARAM_NAMES = ["v_top", "v_bottom", "z0", "k"]
PARAM_BOUNDS = {
    "v_top": (1200.0, 2600.0),
    "v_bottom": (3000.0, 7000.0),
    "z0": (80.0, 420.0),
    "k": (0.005, 0.080),
}
PARAM_RANGES = np.array([1400.0, 4000.0, 340.0, 0.075])


@dataclass
class ModelConfig:
    x_min: float = 0.0
    x_max: float = 1200.0
    z_min: float = 0.0
    z_max: float = 500.0
    nx: int = 51
    nz: int = 41
    source_x: float = 600.0
    receiver_count: int = 21
    receiver_z: float = 0.0


CONFIG = ModelConfig()


# ============================================================
# 2. СЕТКА И ПРЯМАЯ МОДЕЛЬ
# ============================================================

XS = np.linspace(CONFIG.x_min, CONFIG.x_max, CONFIG.nx)
ZS = np.linspace(CONFIG.z_min, CONFIG.z_max, CONFIG.nz)
DX = XS[1] - XS[0]
DZ = ZS[1] - ZS[0]

SOURCE_IX = int(np.argmin(np.abs(XS - CONFIG.source_x)))
SOURCE_IZ_LIST = [int(np.argmin(np.abs(ZS - z))) for z in SOURCE_Z_LIST]

RECEIVER_XS = np.linspace(CONFIG.x_min + 50.0, CONFIG.x_max - 50.0, CONFIG.receiver_count)
RECEIVER_IZ = int(np.argmin(np.abs(ZS - CONFIG.receiver_z)))
RECEIVER_IXS = [int(np.argmin(np.abs(XS - x))) for x in RECEIVER_XS]


def clip_params(params):
    arr = np.asarray(params, dtype=float)
    one = arr.ndim == 1
    if one:
        arr = arr.reshape(1, -1)
    for j, name in enumerate(PARAM_NAMES):
        lo, hi = PARAM_BOUNDS[name]
        arr[:, j] = np.clip(arr[:, j], lo, hi)
    return arr[0] if one else arr


def sigmoid_values(params):
    v_top, v_bottom, z0, k = params
    return v_top + (v_bottom - v_top) / (1.0 + np.exp(-k * (ZS - z0)))


def velocity_model(params):
    vals = sigmoid_values(params)
    return np.repeat(vals[:, None], len(XS), axis=1)


def dijkstra_receivers(vmodel, source_iz):
    nz, nx = vmodel.shape
    T = np.full((nz, nx), np.inf)
    T[source_iz, SOURCE_IX] = 0.0
    heap = [(0.0, source_iz, SOURCE_IX)]
    neigh = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]

    while heap:
        t, iz, ix = heapq.heappop(heap)
        if t > T[iz, ix]:
            continue
        for dz_i, dx_i in neigh:
            nz_i, nx_i = iz + dz_i, ix + dx_i
            if nz_i < 0 or nz_i >= nz or nx_i < 0 or nx_i >= nx:
                continue
            dist = math.sqrt((dx_i * DX) ** 2 + (dz_i * DZ) ** 2)
            vel = 0.5 * (vmodel[iz, ix] + vmodel[nz_i, nx_i])
            nt = t + dist / vel
            if nt < T[nz_i, nx_i]:
                T[nz_i, nx_i] = nt
                heapq.heappush(heap, (nt, nz_i, nx_i))

    return np.array([T[RECEIVER_IZ, ix] for ix in RECEIVER_IXS])


def simulate_curves(params):
    vm = velocity_model(clip_params(params))
    return np.array([dijkstra_receivers(vm, source_iz) for source_iz in SOURCE_IZ_LIST])


def simulate_flat(params):
    return simulate_curves(params).reshape(-1)


# ============================================================
# 3. ДАТАСЕТ И ШУМ
# ============================================================

def sample_params(rng):
    return np.array([
        rng.uniform(*PARAM_BOUNDS["v_top"]),
        rng.uniform(*PARAM_BOUNDS["v_bottom"]),
        rng.uniform(*PARAM_BOUNDS["z0"]),
        rng.uniform(*PARAM_BOUNDS["k"]),
    ], dtype=float)


def make_dataset(n, seed):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for i in range(n):
        p = sample_params(rng)
        X.append(simulate_flat(p))
        y.append(p)
        if (i + 1) % 50 == 0:
            print(f"Сгенерировано {i + 1}/{n} профилей...")
    return np.array(X), np.array(y)


def add_noise(X, rel_noise, seed):
    X = np.asarray(X)
    if rel_noise == 0:
        return X.copy()
    rng = np.random.default_rng(seed)
    return np.maximum(X + rng.normal(0.0, rel_noise * X, size=X.shape), 1e-6)


def augment_train(X_train, y_train, levels, seed):
    Xs, ys = [], []
    for i, nl in enumerate(levels):
        Xs.append(add_noise(X_train, nl, seed + i * 1000 + int(nl * 100000)))
        ys.append(y_train.copy())
    return np.vstack(Xs), np.vstack(ys)


# ============================================================
# 4. MULTI-SOURCE ПРИЗНАКИ
# ============================================================

def reshape_sources(Xflat):
    Xflat = np.asarray(Xflat)
    return Xflat.reshape(Xflat.shape[0], len(SOURCE_Z_LIST), CONFIG.receiver_count)


def features(Xflat):
    Xs = reshape_sources(Xflat)
    parts = []
    n_src = Xs.shape[1]
    n_rec = Xs.shape[2]
    c = n_rec // 2

    for s in range(n_src):
        X = Xs[:, s, :]
        raw = X
        mn = X.min(axis=1, keepdims=True)
        rg = np.maximum(X.max(axis=1, keepdims=True) - mn, 1e-9)
        norm = (X - mn) / rg
        d1 = np.gradient(X, axis=1)
        summ = np.column_stack([
            X.min(axis=1), X.max(axis=1), X.mean(axis=1), X.std(axis=1),
            X.max(axis=1) - X.min(axis=1), X[:, c],
            X[:, c] - X[:, 0], X[:, -1] - X[:, c],
        ])
        parts += [raw, norm, d1, summ]

    for s in range(n_src - 1):
        diff = Xs[:, s + 1, :] - Xs[:, s, :]
        diff_summ = np.column_stack([
            diff.min(axis=1), diff.max(axis=1), diff.mean(axis=1),
            diff.std(axis=1), diff.max(axis=1) - diff.min(axis=1),
        ])
        parts += [diff, diff_summ]

    return np.hstack(parts)


def feature_names():
    names = []
    for s, z in enumerate(SOURCE_Z_LIST):
        lab = f"src{s}_z{int(z)}"
        names += [f"{lab}_t{i+1}" for i in range(CONFIG.receiver_count)]
        names += [f"{lab}_tnorm{i+1}" for i in range(CONFIG.receiver_count)]
        names += [f"{lab}_d1_{i+1}" for i in range(CONFIG.receiver_count)]
        names += [
            f"{lab}_t_min", f"{lab}_t_max", f"{lab}_t_mean", f"{lab}_t_std",
            f"{lab}_t_range", f"{lab}_t_center", f"{lab}_left_slope", f"{lab}_right_slope"
        ]
    for s in range(len(SOURCE_Z_LIST)-1):
        lab = f"diff_src{s+1}_minus_src{s}"
        names += [f"{lab}_{i+1}" for i in range(CONFIG.receiver_count)]
        names += [f"{lab}_min", f"{lab}_max", f"{lab}_mean", f"{lab}_std", f"{lab}_range"]
    return names


# ============================================================
# 5. РЕГРЕССОР
# ============================================================

def train_regressor(Xf, y):
    ys = StandardScaler()
    y_scaled = ys.fit_transform(y)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=2,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(Xf, y_scaled)
    return model, ys


def predict_regressor(model, ys, Xf):
    return clip_params(ys.inverse_transform(model.predict(Xf)))


# ============================================================
# 6. LOSS, RANDOM SEARCH, LOCAL REFINEMENT
# ============================================================

def rmse_ms(a, b):
    return math.sqrt(mean_squared_error(a, b)) * 1000.0


def loss_curve_ms(params, observed_flat):
    return rmse_ms(simulate_flat(params), observed_flat)


def random_search(observed_flat, n_trials, seed):
    rng = np.random.default_rng(seed)
    best_p, best_l = None, np.inf
    for _ in range(n_trials):
        p = sample_params(rng)
        l = loss_curve_ms(p, observed_flat)
        if l < best_l:
            best_p, best_l = p, l
    return clip_params(best_p), best_l


def coordinate_refine(start, observed_flat, rounds=10):
    p = clip_params(start)
    steps = np.array([60.0, 180.0, 18.0, 0.004])
    best = loss_curve_ms(p, observed_flat)
    hist = []
    for r in range(rounds):
        improved = False
        for j in range(4):
            for sign in [-1, 1]:
                cand = p.copy()
                cand[j] += sign * steps[j]
                cand = clip_params(cand)
                l = loss_curve_ms(cand, observed_flat)
                if l < best:
                    p, best, improved = cand, l, True
        hist.append({"round": r, "loss_ms": best, "v_top": p[0], "v_bottom": p[1], "z0": p[2], "k": p[3]})
        steps *= 0.80 if improved else 0.55
    return p, best, pd.DataFrame(hist)


def multistart_local(start, observed_flat, seed, n_starts):
    rng = np.random.default_rng(seed)
    starts = [clip_params(start)]
    scales = np.array([90.0, 250.0, 30.0, 0.007])
    for _ in range(n_starts - 1):
        starts.append(clip_params(start + rng.normal(0, scales)))
    best_p, best_l, best_h = None, np.inf, None
    for st in starts:
        p, l, h = coordinate_refine(st, observed_flat)
        if l < best_l:
            best_p, best_l, best_h = p, l, h
    return best_p, best_l, best_h


# ============================================================
# 7. MCMC
# ============================================================

def log_prior(p):
    for j, name in enumerate(PARAM_NAMES):
        lo, hi = PARAM_BOUNDS[name]
        if p[j] < lo or p[j] > hi:
            return -np.inf
    return 0.0


def logpost_and_loss(p, observed_flat, noise_level):
    """
    Возвращает log posterior и curve RMSE за один расчет прямой задачи.

    Это важно для скорости: MCMC дорогой, потому что каждый шаг требует
    повторного решения прямой задачи для трех источников.
    """
    p = np.asarray(p, dtype=float)
    lp = log_prior(p)
    if not np.isfinite(lp):
        return -np.inf, np.inf

    pred = simulate_flat(p)
    resid = observed_flat - pred

    # Минимальный floor нужен для clean case, иначе likelihood становится
    # слишком жестким и цепь почти не двигается.
    sigma = max(0.0015, noise_level * np.median(observed_flat))

    logp = lp - 0.5 * np.sum((resid / sigma) ** 2)
    loss = rmse_ms(pred, observed_flat)
    return logp, loss


def log_post(p, observed_flat, noise_level):
    logp, _ = logpost_and_loss(p, observed_flat, noise_level)
    return logp


def run_mcmc(start, observed_flat, noise_level, seed):
    rng = np.random.default_rng(seed)
    cur = clip_params(start)
    cur_lp, cur_loss = logpost_and_loss(cur, observed_flat, noise_level)

    # Старт уже хороший: ExtraTrees + local refinement. Поэтому proposal умеренный.
    scales = np.array([35.0, 100.0, 10.0, 0.0025])
    adapt_every = 50

    chain, logps, losses, acc_flags = [], [], [], []
    best_p, best_lp, best_loss = cur.copy(), cur_lp, cur_loss
    acc = 0
    win_acc = 0
    win_total = 0

    for step in range(MCMC_STEPS):
        cand = clip_params(cur + rng.normal(0, scales))
        cand_lp, cand_loss = logpost_and_loss(cand, observed_flat, noise_level)

        accept = False
        if np.isfinite(cand_lp) and math.log(rng.uniform()) < cand_lp - cur_lp:
            accept = True

        if accept:
            cur, cur_lp, cur_loss = cand, cand_lp, cand_loss
            acc += 1
            win_acc += 1

        win_total += 1

        if cur_lp > best_lp:
            best_p, best_lp, best_loss = cur.copy(), cur_lp, cur_loss

        chain.append(cur.copy())
        logps.append(cur_lp)
        losses.append(cur_loss)
        acc_flags.append(int(accept))

        # Практическая адаптация только во время burn-in.
        # В предыдущей версии условие было слишком жестким и в FAST почти не срабатывало.
        if step < MCMC_BURN_IN and (step + 1) % adapt_every == 0:
            ar = win_acc / max(1, win_total)
            if ar < 0.10:
                scales *= 0.75
            elif ar > 0.40:
                scales *= 1.20
            win_acc, win_total = 0, 0

    chain = np.array(chain)
    logps = np.array(logps)
    losses = np.array(losses)
    acc_flags = np.array(acc_flags)
    post = chain[MCMC_BURN_IN::MCMC_THIN]

    # На случай, если пользователь слишком сильно уменьшит MCMC_STEPS вручную.
    if len(post) == 0:
        post = chain

    return {
        "best_params": clip_params(best_p),
        "posterior_median": clip_params(np.median(post, axis=0)),
        "posterior_mean": clip_params(np.mean(post, axis=0)),
        "q05": np.quantile(post, 0.05, axis=0),
        "q95": np.quantile(post, 0.95, axis=0),
        "acceptance_rate": acc / MCMC_STEPS,
        "best_loss_ms": best_loss,
        "chain": chain,
        "logps": logps,
        "losses": losses,
        "accepted": acc_flags,
        "posterior_chain": post,
    }


# ============================================================
# 8. МЕТРИКИ И ГРАФИКИ
# ============================================================

def param_metric_rows(y_true, y_pred, method, noise):
    y_pred = clip_params(y_pred)
    rows = []
    for j, name in enumerate(PARAM_NAMES):
        rows.append({
            "method": method,
            "noise_level": noise,
            "parameter": name,
            "MAE": mean_absolute_error(y_true[:, j], y_pred[:, j]),
            "RMSE": math.sqrt(mean_squared_error(y_true[:, j], y_pred[:, j])),
            "R2": r2_score(y_true[:, j], y_pred[:, j]) if len(y_true) > 1 else np.nan,
            "normalized_MAE": mean_absolute_error(y_true[:, j], y_pred[:, j]) / PARAM_RANGES[j],
        })
    return rows


def compare_methods(y_true, clean_flats, observed_flats, pred_dict, noise):
    p_rows, c_rows = [], []
    for method, preds in pred_dict.items():
        preds = clip_params(np.asarray(preds))
        p_rows += param_metric_rows(y_true, preds, method, noise)
        for i, p in enumerate(preds):
            pf = simulate_flat(p)
            c_rows.append({
                "method": method,
                "noise_level": noise,
                "sample_index": i,
                "predicted_vs_observed_curve_RMSE_ms": rmse_ms(pf, observed_flats[i]),
                "predicted_vs_clean_curve_RMSE_ms": rmse_ms(pf, clean_flats[i]),
                "observed_vs_clean_curve_RMSE_ms": rmse_ms(observed_flats[i], clean_flats[i]),
            })
    return pd.DataFrame(p_rows), pd.DataFrame(c_rows)


def save_bar(df, col, fname):
    s = df.groupby("method", as_index=False)[col].mean().sort_values(col)
    plt.figure(figsize=(9,5))
    plt.bar(s["method"], s[col])
    plt.xticks(rotation=25, ha="right")
    plt.ylabel(col)
    plt.title(f"Method comparison: {col}")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(fname, dpi=200, bbox_inches="tight")
    plt.show()


def plot_profile(true_p, rand_p, reg_p, loc_p, mcmc_p, fname):
    plt.figure(figsize=(6,6))
    plt.plot(sigmoid_values(true_p), ZS, label="true")
    plt.plot(sigmoid_values(rand_p), ZS, "--", label="random search")
    plt.plot(sigmoid_values(reg_p), ZS, "--", label="ExtraTrees")
    plt.plot(sigmoid_values(loc_p), ZS, ":", label="ExtraTrees + local")
    plt.plot(sigmoid_values(mcmc_p), ZS, "-.", label="ExtraTrees + MCMC")
    plt.gca().invert_yaxis()
    plt.xlabel("Скорость, м/с")
    plt.ylabel("Глубина, м")
    plt.title("Demo profile comparison")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(fname, dpi=200, bbox_inches="tight")
    plt.show()


def plot_curves(clean, obs, rand, reg, loc, mcmc, prefix):
    mats = {
        "clean": clean.reshape(len(SOURCE_Z_LIST), CONFIG.receiver_count),
        "observed": obs.reshape(len(SOURCE_Z_LIST), CONFIG.receiver_count),
        "random": rand.reshape(len(SOURCE_Z_LIST), CONFIG.receiver_count),
        "ExtraTrees": reg.reshape(len(SOURCE_Z_LIST), CONFIG.receiver_count),
        "local": loc.reshape(len(SOURCE_Z_LIST), CONFIG.receiver_count),
        "MCMC": mcmc.reshape(len(SOURCE_Z_LIST), CONFIG.receiver_count),
    }
    for s, z in enumerate(SOURCE_Z_LIST):
        plt.figure(figsize=(9,5))
        for name, mat in mats.items():
            plt.plot(RECEIVER_XS, mat[s], marker="o", label=name)
        plt.xlabel("x приемника, м")
        plt.ylabel("Время прихода, с")
        plt.title(f"Demo curves, source_z={int(z)} m")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        fname = f"{prefix}_source_z_{int(z)}.png"
        plt.savefig(fname, dpi=200, bbox_inches="tight")
        plt.show()


def plot_mcmc_diagnostics(res, true_p):
    plt.figure(figsize=(9,5))
    plt.plot(res["losses"])
    plt.xlabel("MCMC step")
    plt.ylabel("Curve RMSE, ms")
    plt.title("MCMC loss history")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("stage10_checked_demo_mcmc_loss_history.png", dpi=200, bbox_inches="tight")
    plt.show()

    chain = res["chain"]
    post = res["posterior_chain"]
    for j, name in enumerate(PARAM_NAMES):
        plt.figure(figsize=(9,4))
        plt.plot(chain[:, j])
        plt.axhline(true_p[j], linestyle="--", label="true")
        plt.xlabel("MCMC step")
        plt.ylabel(name)
        plt.title(f"MCMC trace: {name}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"stage10_checked_demo_mcmc_trace_{name}.png", dpi=200, bbox_inches="tight")
        plt.show()

        plt.figure(figsize=(7,4))
        plt.hist(post[:, j], bins=30)
        plt.axvline(true_p[j], linestyle="--", label="true")
        plt.xlabel(name)
        plt.ylabel("count")
        plt.title(f"MCMC posterior: {name}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"stage10_checked_demo_mcmc_posterior_{name}.png", dpi=200, bbox_inches="tight")
        plt.show()


# ============================================================
# 9. ЗАПУСК ЭТАПА 10
# ============================================================

print("=" * 90)
print("STAGE 10 CHECKED: HYBRID MULTI-SOURCE INVERSE BENCHMARK")
print("=" * 90)
print("RUN_MODE:", RUN_MODE)
print("N_BASE_SAMPLES:", N_BASE_SAMPLES)
print("BENCHMARK_SAMPLES:", BENCHMARK_SAMPLES)
print("MCMC_STEPS:", MCMC_STEPS)

print("\nГенерируем датасет...")
X, y = make_dataset(N_BASE_SAMPLES, RANDOM_SEED)

cols = [f"source{s}_z{int(z)}_t{i+1}" for s, z in enumerate(SOURCE_Z_LIST) for i in range(CONFIG.receiver_count)]
df_dataset = pd.DataFrame(X, columns=cols)
for j, name in enumerate(PARAM_NAMES):
    df_dataset[name] = y[:, j]
df_dataset.to_csv("stage10_checked_multisource_dataset.csv", index=False)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_SEED)
X_train_aug, y_train_aug = augment_train(X_train, y_train, TRAIN_NOISE_LEVELS, RANDOM_SEED + 1000)

print("Train clean:", X_train.shape)
print("Train augmented:", X_train_aug.shape)
print("Test:", X_test.shape)

print("\nОбучаем ExtraTreesRegressor...")
model, y_scaler = train_regressor(features(X_train_aug), y_train_aug)

# Feature importance
fnames = feature_names()
imp = model.feature_importances_
idx = np.argsort(imp)[::-1][:30]
plt.figure(figsize=(11,5))
plt.bar(range(len(idx)), imp[idx])
plt.xticks(range(len(idx)), [fnames[i] for i in idx], rotation=60, ha="right")
plt.ylabel("Importance")
plt.title("ExtraTrees feature importance")
plt.tight_layout()
plt.savefig("stage10_checked_extratrees_feature_importance.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# 10. FULL TEST: только регрессор
# ============================================================

full_param_rows, full_curve_rows = [], []
for nl in FULL_TEST_NOISE_LEVELS:
    print(f"\nFull ExtraTrees test, noise={nl*100:.1f}%")
    Xobs = add_noise(X_test, nl, RANDOM_SEED + 2000 + int(nl * 100000))
    pred = predict_regressor(model, y_scaler, features(Xobs))
    full_param_rows += param_metric_rows(y_test, pred, "ExtraTrees_only_full_test", nl)

    subset = min(100, len(X_test))
    for i in range(subset):
        pf = simulate_flat(pred[i])
        full_curve_rows.append({
            "method": "ExtraTrees_only_full_test",
            "noise_level": nl,
            "sample_index": i,
            "predicted_vs_observed_curve_RMSE_ms": rmse_ms(pf, Xobs[i]),
            "predicted_vs_clean_curve_RMSE_ms": rmse_ms(pf, X_test[i]),
            "observed_vs_clean_curve_RMSE_ms": rmse_ms(Xobs[i], X_test[i]),
        })

full_param_df = pd.DataFrame(full_param_rows)
full_curve_df = pd.DataFrame(full_curve_rows)
full_param_df.to_csv("stage10_checked_full_regressor_parameter_errors.csv", index=False)
full_curve_df.to_csv("stage10_checked_full_regressor_curve_rmse.csv", index=False)
display(full_param_df)

# ============================================================
# 11. BENCHMARK: все методы на subset
# ============================================================

bench_idx = np.arange(min(BENCHMARK_SAMPLES, len(X_test)))
X_clean_b = X_test[bench_idx]
y_true_b = y_test[bench_idx]

bench_param_parts, bench_curve_parts, mcmc_info = [], [], []

for nl in BENCHMARK_NOISE_LEVELS:
    print("\n" + "=" * 90)
    print(f"BENCHMARK noise={nl*100:.1f}%")
    print("=" * 90)

    Xobs_b = add_noise(X_clean_b, nl, RANDOM_SEED + 3000 + int(nl * 100000))
    reg_pred = predict_regressor(model, y_scaler, features(Xobs_b))

    rand_pred, loc_pred, mcmc_pred = [], [], []
    for i in range(len(Xobs_b)):
        print(f"sample {i+1}/{len(Xobs_b)}")

        obs = Xobs_b[i]

        rp, rl = random_search(obs, RANDOM_SEARCH_TRIALS, RANDOM_SEED + 4000 + i + int(nl * 100000))
        rand_pred.append(rp)

        lp, ll, lh = multistart_local(reg_pred[i], obs, RANDOM_SEED + 5000 + i + int(nl * 100000), LOCAL_STARTS)
        loc_pred.append(lp)

        # MCMC стартует от локально уточненного решения.
        mres = run_mcmc(lp, obs, nl, RANDOM_SEED + 6000 + i + int(nl * 100000))
        mcmc_pred.append(mres["best_params"])

        mcmc_info.append({
            "noise_level": nl,
            "sample_index": i,
            "acceptance_rate": mres["acceptance_rate"],
            "best_loss_ms": mres["best_loss_ms"],
            "posterior_median_v_top": mres["posterior_median"][0],
            "posterior_median_v_bottom": mres["posterior_median"][1],
            "posterior_median_z0": mres["posterior_median"][2],
            "posterior_median_k": mres["posterior_median"][3],
        })

    pred_dict = {
        "random_search_baseline": np.array(rand_pred),
        "ExtraTrees_only": np.array(reg_pred),
        "ExtraTrees_plus_local": np.array(loc_pred),
        "ExtraTrees_plus_local_plus_MCMC_MAP": np.array(mcmc_pred),
    }
    p_df, c_df = compare_methods(y_true_b, X_clean_b, Xobs_b, pred_dict, nl)
    bench_param_parts.append(p_df)
    bench_curve_parts.append(c_df)

    display(p_df)
    display(c_df.groupby("method", as_index=False).agg(
        mean_predicted_vs_observed_curve_RMSE_ms=("predicted_vs_observed_curve_RMSE_ms", "mean"),
        mean_predicted_vs_clean_curve_RMSE_ms=("predicted_vs_clean_curve_RMSE_ms", "mean"),
    ))

bench_param_df = pd.concat(bench_param_parts, ignore_index=True)
bench_curve_df = pd.concat(bench_curve_parts, ignore_index=True)
mcmc_info_df = pd.DataFrame(mcmc_info)

bench_param_df.to_csv("stage10_checked_benchmark_parameter_errors.csv", index=False)
bench_curve_df.to_csv("stage10_checked_benchmark_curve_rmse.csv", index=False)
mcmc_info_df.to_csv("stage10_checked_benchmark_mcmc_info.csv", index=False)

overall_param = bench_param_df.groupby(["method", "noise_level"], as_index=False).agg(
    mean_normalized_MAE=("normalized_MAE", "mean"),
    mean_R2=("R2", "mean"),
)
overall_curve = bench_curve_df.groupby(["method", "noise_level"], as_index=False).agg(
    mean_predicted_vs_observed_curve_RMSE_ms=("predicted_vs_observed_curve_RMSE_ms", "mean"),
    mean_predicted_vs_clean_curve_RMSE_ms=("predicted_vs_clean_curve_RMSE_ms", "mean"),
    mean_observed_vs_clean_curve_RMSE_ms=("observed_vs_clean_curve_RMSE_ms", "mean"),
)
overall = overall_param.merge(overall_curve, on=["method", "noise_level"], how="left")
overall.to_csv("stage10_checked_benchmark_overall_by_method.csv", index=False)

print("\nOVERALL BENCHMARK")
display(overall)

save_bar(overall, "mean_normalized_MAE", "stage10_checked_benchmark_bar_mean_normalized_MAE.png")
save_bar(overall, "mean_predicted_vs_observed_curve_RMSE_ms", "stage10_checked_benchmark_bar_curve_RMSE_observed.png")

# ============================================================
# 12. DEMO: posterior/trace/curves/profile
# ============================================================

print("\n" + "=" * 90)
print("DEMO")
print("=" * 90)

demo_i = 0
true_p = y_true_b[demo_i]
clean = X_clean_b[demo_i]
obs = add_noise(clean.reshape(1, -1), DEMO_NOISE_LEVEL, RANDOM_SEED + 8888)[0]

reg_p = predict_regressor(model, y_scaler, features(obs.reshape(1, -1)))[0]
rand_p, _ = random_search(obs, RANDOM_SEARCH_TRIALS, RANDOM_SEED + 9000)
loc_p, _, loc_hist = multistart_local(reg_p, obs, RANDOM_SEED + 9100, LOCAL_STARTS)
mres = run_mcmc(loc_p, obs, DEMO_NOISE_LEVEL, RANDOM_SEED + 9200)
mcmc_p = mres["best_params"]

methods = {
    "true": true_p,
    "random_search_baseline": rand_p,
    "ExtraTrees_only": reg_p,
    "ExtraTrees_plus_local": loc_p,
    "ExtraTrees_plus_local_plus_MCMC_MAP": mcmc_p,
    "MCMC_posterior_median": mres["posterior_median"],
}

demo_param_rows = []
for method, p in methods.items():
    for j, name in enumerate(PARAM_NAMES):
        demo_param_rows.append({
            "method": method,
            "parameter": name,
            "value": p[j],
            "true_value": true_p[j],
            "absolute_error": abs(p[j] - true_p[j]),
        })
demo_param_df = pd.DataFrame(demo_param_rows)
demo_param_df.to_csv("stage10_checked_demo_parameter_summary.csv", index=False)

demo_curve_rows = []
for method, p in methods.items():
    pred_flat = clean if method == "true" else simulate_flat(p)
    demo_curve_rows.append({
        "method": method,
        "predicted_vs_observed_curve_RMSE_ms": rmse_ms(pred_flat, obs),
        "predicted_vs_clean_curve_RMSE_ms": rmse_ms(pred_flat, clean),
    })
demo_curve_df = pd.DataFrame(demo_curve_rows)
demo_curve_df.to_csv("stage10_checked_demo_curve_rmse_summary.csv", index=False)

chain_df = pd.DataFrame(mres["chain"], columns=PARAM_NAMES)
chain_df["log_posterior"] = mres["logps"]
chain_df["curve_RMSE_ms"] = mres["losses"]
chain_df["accepted"] = mres["accepted"]
chain_df.to_csv("stage10_checked_demo_mcmc_chain.csv", index=False)

post_rows = []
for j, name in enumerate(PARAM_NAMES):
    post_rows.append({
        "parameter": name,
        "true": true_p[j],
        "mcmc_best_MAP": mcmc_p[j],
        "posterior_mean": mres["posterior_mean"][j],
        "posterior_median": mres["posterior_median"][j],
        "q05": mres["q05"][j],
        "q95": mres["q95"][j],
        "true_inside_90_interval": bool(mres["q05"][j] <= true_p[j] <= mres["q95"][j]),
    })
post_df = pd.DataFrame(post_rows)
post_df["acceptance_rate"] = mres["acceptance_rate"]
post_df.to_csv("stage10_checked_demo_mcmc_posterior_summary.csv", index=False)

loc_hist.to_csv("stage10_checked_demo_local_refinement_history.csv", index=False)

display(demo_param_df)
display(demo_curve_df)
display(post_df)

plot_profile(true_p, rand_p, reg_p, loc_p, mcmc_p, "stage10_checked_demo_profile_comparison.png")
plot_curves(
    clean=clean,
    obs=obs,
    rand=simulate_flat(rand_p),
    reg=simulate_flat(reg_p),
    loc=simulate_flat(loc_p),
    mcmc=simulate_flat(mcmc_p),
    prefix="stage10_checked_demo_curve_comparison",
)
plot_mcmc_diagnostics(mres, true_p)

print("\nГотово. Чтобы скачать все результаты, выполни:")
print("download_stage10_checked_results()")


# ============================================================
# 13. СКАЧИВАНИЕ РЕЗУЛЬТАТОВ
# ============================================================

def download_stage10_checked_results():
    import zipfile
    from pathlib import Path
    from google.colab import files

    base = Path("/content")
    zip_path = base / "stage10_checked_hybrid_inverse_results.zip"

    result_files = []
    for path in base.rglob("*"):
        if path.is_file() and path.name.startswith("stage10_checked_") and (
            path.name.endswith(".csv") or path.name.endswith(".png")
        ):
            result_files.append(path)

    result_files = sorted(set(result_files))
    print("Найдено файлов:", len(result_files))
    for p in result_files:
        print(p.name)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in result_files:
            z.write(p, arcname=p.name)

    print("Архив создан:", zip_path)
    files.download(str(zip_path))


## Этап 10.5 — ансамбль MCMC

Stage 10.5: lightweight ensemble MCMC demo.
Purpose: close the project requirements: Metropolis-Hastings, Robbins-Monro style
proposal adaptation, likelihood, ensemble of chains, MAP selection, convergence plots.

In [ ]:
from pathlib import Path
import os, math, heapq, zipfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

RESULTS_DIR = Path('/content/stage10_5_ensemble_mcmc_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(RESULTS_DIR)

SEED=42; rng_global=np.random.default_rng(SEED)
SOURCE_Z=[250.,350.,450.]
PARAMS=['v_top','v_bottom','z0','k']
BOUNDS={'v_top':(1200.,2600.),'v_bottom':(3000.,7000.),'z0':(80.,420.),'k':(0.005,0.080)}
RANGES=np.array([BOUNDS[p][1]-BOUNDS[p][0] for p in PARAMS])
NX,NZ=51,41; XMIN,XMAX,ZMIN,ZMAX=0.,1200.,0.,500.; SRC_X=600.; NREC=21
XS=np.linspace(XMIN,XMAX,NX); ZS=np.linspace(ZMIN,ZMAX,NZ); DX=XS[1]-XS[0]; DZ=ZS[1]-ZS[0]
SRC_IX=int(np.argmin(abs(XS-SRC_X))); SRC_IZ=[int(np.argmin(abs(ZS-z))) for z in SOURCE_Z]
REC_X=np.linspace(XMIN+50,XMAX-50,NREC); REC_IZ=int(np.argmin(abs(ZS-0.)))
REC_IX=[int(np.argmin(abs(XS-x))) for x in REC_X]
N_FALLBACK=1200; N_TREES=300; TRAIN_NOISE=[0,.005,.01,.02,.05]
DEMO_NOISE=.01; N_CHAINS=4; STEPS=700; BURN=200; THIN=2; TARGET=.25

def clip(p):
    a=np.array(p,dtype=float).copy(); one=a.ndim==1
    if one: a=a.reshape(1,-1)
    for i,n in enumerate(PARAMS): a[:,i]=np.clip(a[:,i],*BOUNDS[n])
    return a[0] if one else a

def v_profile(p):
    vt,vb,z0,k=p
    return np.array([vt+(vb-vt)/(1+math.exp(-k*(z-z0))) for z in ZS])

def vel_model(p):
    prof=v_profile(p); return np.repeat(prof[:,None], NX, axis=1)

def dijkstra_times(vm, src_iz):
    T=np.full((NZ,NX),np.inf); T[src_iz,SRC_IX]=0.; heap=[(0.,src_iz,SRC_IX)]
    neigh=[(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
    while heap:
        t,iz,ix=heapq.heappop(heap)
        if t>T[iz,ix]: continue
        for dz,dx in neigh:
            jz,jx=iz+dz,ix+dx
            if 0<=jz<NZ and 0<=jx<NX:
                dist=math.sqrt((dx*DX)**2+(dz*DZ)**2); v=.5*(vm[iz,ix]+vm[jz,jx])
                nt=t+dist/v
                if nt<T[jz,jx]: T[jz,jx]=nt; heapq.heappush(heap,(nt,jz,jx))
    return np.array([T[REC_IZ,ix] for ix in REC_IX])

def sim_curves(p):
    vm=vel_model(p); return np.array([dijkstra_times(vm,iz) for iz in SRC_IZ])

def sim_flat(p): return sim_curves(p).reshape(-1)

def sample_params(rng): return np.array([rng.uniform(*BOUNDS[n]) for n in PARAMS])

def data_cols(): return [f'source{s}_z{int(z)}_t{i+1}' for s,z in enumerate(SOURCE_Z) for i in range(NREC)]
COLS=data_cols()

def load_or_generate():
    cands=[Path('/content/stage9_1_checked_multisource_regression_dataset.csv'),Path('/content/stage10_checked_multisource_dataset.csv'),Path('/content/stage10_fast_check/stage10_checked_multisource_dataset.csv'),Path('/content/stage10_accurate_results/stage10_checked_multisource_dataset.csv')]
    for p in cands:
        if p.exists():
            df=pd.read_csv(p)
            if all(c in df.columns for c in COLS+PARAMS):
                print('Loaded cached dataset:',p,df.shape)
                return df[COLS].to_numpy(float), df[PARAMS].to_numpy(float)
    print('No cached dataset found. Generating fallback dataset...')
    r=np.random.default_rng(SEED); X=[]; y=[]
    for i in range(N_FALLBACK):
        p=sample_params(r); X.append(sim_flat(p)); y.append(p)
        if (i+1)%50==0: print('generated',i+1,'/',N_FALLBACK)
    return np.array(X),np.array(y)

def add_noise(X, level, seed):
    if level==0: return X.copy()
    r=np.random.default_rng(seed); return np.maximum(X+r.normal(0,level*X,size=X.shape),1e-6)

def features(Xflat):
    A=np.asarray(Xflat).reshape(len(Xflat),len(SOURCE_Z),NREC); parts=[]
    for s in range(len(SOURCE_Z)):
        X=A[:,s,:]; mn=X.min(1,keepdims=True); rg=np.maximum(X.max(1,keepdims=True)-mn,1e-9)
        norm=(X-mn)/rg; d1=np.gradient(X,axis=1); c=NREC//2
        summ=np.c_[X.min(1),X.max(1),X.mean(1),X.std(1),X.max(1)-X.min(1),X[:,c],X[:,c]-X[:,0],X[:,-1]-X[:,c]]
        parts += [X,norm,d1,summ]
    for s in range(len(SOURCE_Z)-1):
        D=A[:,s+1,:]-A[:,s,:]
        parts += [D, np.c_[D.min(1),D.max(1),D.mean(1),D.std(1),D.max(1)-D.min(1)]]
    return np.hstack(parts)

def train_model(X,y):
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEED)
    Xa=[]; ya=[]
    for i,lv in enumerate(TRAIN_NOISE): Xa.append(add_noise(Xtr,lv,SEED+1000+i)); ya.append(ytr)
    Xa=np.vstack(Xa); ya=np.vstack(ya)
    sc=StandardScaler(); ys=sc.fit_transform(ya)
    m=ExtraTreesRegressor(n_estimators=N_TREES,min_samples_leaf=2,random_state=SEED,n_jobs=-1)
    print('Training ExtraTrees:',Xa.shape,features(Xa).shape)
    m.fit(features(Xa),ys)
    return m,sc,Xte,yte

def predict(m,sc,X): return clip(sc.inverse_transform(m.predict(features(X))))

def rmse_flat(a,b): return math.sqrt(mean_squared_error(a,b))*1000

def loss(p,obs): return rmse_flat(sim_flat(p),obs)

def local_refine(p0,obs,rounds=10):
    p=clip(p0); steps=np.array([60.,180.,18.,.004]); best=loss(p,obs); rows=[]
    for r in range(rounds):
        improved=False
        for j in range(4):
            for d in [-1,1]:
                q=clip(p+np.eye(4)[j]*steps[j]*d); L=loss(q,obs)
                if L<best: p,best,improved=q,L,True
        rows.append(dict(round=r,loss_ms=best,v_top=p[0],v_bottom=p[1],z0=p[2],k=p[3]))
        steps*=.8 if improved else .55
    return p,best,pd.DataFrame(rows)

def logprior(p):
    return 0. if all(BOUNDS[n][0]<=p[i]<=BOUNDS[n][1] for i,n in enumerate(PARAMS)) else -np.inf

def loglike(p,obs,noise):
    if not np.isfinite(logprior(p)): return -np.inf
    pred=sim_flat(p); sig=max(.0015, noise*np.median(obs)); res=obs-pred
    return -.5*np.sum((res/sig)**2)

def logpost(p,obs,noise): return logprior(p)+loglike(p,obs,noise)

def run_chain(cid,start,obs,noise,steps=STEPS,burn=BURN,target=TARGET,seed=SEED):
    r=np.random.default_rng(seed+cid*777); cur=clip(start); logp=logpost(cur,obs,noise); cur_loss=loss(cur,obs)
    base=np.array([35.,100.,10.,.0025]); logmult=0.; acc=0; rows=[]; best=(logp,cur.copy(),cur_loss,0)
    for t in range(steps):
        q=clip(cur+r.normal(0,base*math.exp(logmult))); lq=logpost(q,obs,noise); accepted=0
        if np.isfinite(lq) and math.log(r.uniform())<lq-logp:
            cur,logp=q,lq; cur_loss=loss(cur,obs); accepted=1; acc+=1
        if logp>best[0]: best=(logp,cur.copy(),cur_loss,t)
        if t<burn:
            gamma=.08/math.sqrt(t+1); logmult += gamma*(accepted-target)  # Robbins-Monro style adaptation
        rows.append(dict(chain_id=cid,step=t,v_top=cur[0],v_bottom=cur[1],z0=cur[2],k=cur[3],log_posterior=logp,curve_RMSE_ms=cur_loss,accepted=accepted,proposal_scale_multiplier=math.exp(logmult)))
    df=pd.DataFrame(rows); return df, best, acc/steps

def run_ensemble(start,obs,noise):
    r=np.random.default_rng(SEED+7000); starts=[]; scale=np.array([50.,150.,15.,.004])
    for cid in range(N_CHAINS): starts.append(clip(start if cid==0 else start+r.normal(0,scale)))
    dfs=[]; diag=[]; bests=[]
    for cid,st in enumerate(starts):
        print('MCMC chain',cid+1,'/',N_CHAINS)
        df,best,ar=run_chain(cid,st,obs,noise,seed=SEED+9000); dfs.append(df); bests.append(best)
        diag.append(dict(chain_id=cid,acceptance_rate=ar,best_log_posterior=best[0],best_curve_RMSE_ms=best[2],best_step=best[3]))
    allc=pd.concat(dfs,ignore_index=True); post=allc.groupby('chain_id',group_keys=False).apply(lambda d: d.iloc[BURN::THIN]).reset_index(drop=True)
    idx=allc.log_posterior.idxmax(); row=allc.loc[idx]; map_p=row[PARAMS].to_numpy(float)
    arr=post[PARAMS].to_numpy(float)
    return dict(all=allc,post=post,diag=pd.DataFrame(diag),map=clip(map_p),median=clip(np.median(arr,0)),mean=clip(arr.mean(0)),q05=np.quantile(arr,.05,0),q95=np.quantile(arr,.95,0),map_row=row)

X,y=load_or_generate(); pd.DataFrame(np.c_[X,y],columns=COLS+PARAMS).to_csv('stage10_5_checked_multisource_dataset_used.csv',index=False)
model,scaler,Xtest,ytest=train_model(X,y)
true_p=ytest[0]; clean=Xtest[0]; obs=add_noise(clean.reshape(1,-1),DEMO_NOISE,SEED+555)[0]
reg_p=predict(model,scaler,obs.reshape(1,-1))[0]
loc_p,loc_loss,loc_hist=local_refine(reg_p,obs); loc_hist.to_csv('stage10_5_checked_local_refinement_history.csv',index=False)
ens=run_ensemble(loc_p,obs,DEMO_NOISE)
map_p=ens['map']; med_p=ens['median']
ens['all'].to_csv('stage10_5_checked_ensemble_mcmc_all_chains.csv',index=False)
ens['post'].to_csv('stage10_5_checked_ensemble_mcmc_posterior_samples.csv',index=False)
ens['diag'].to_csv('stage10_5_checked_ensemble_mcmc_chain_diagnostics.csv',index=False)

methods={'true':true_p,'ExtraTrees_initial':reg_p,'ExtraTrees_plus_local':loc_p,'ensemble_MCMC_MAP':map_p,'ensemble_MCMC_posterior_median':med_p}
rows=[]
for method,p in methods.items():
    for i,n in enumerate(PARAMS): rows.append(dict(method=method,parameter=n,value=p[i],true_value=true_p[i],absolute_error=abs(p[i]-true_p[i]),normalized_abs_error=abs(p[i]-true_p[i])/RANGES[i]))
pd.DataFrame(rows).to_csv('stage10_5_checked_demo_parameter_summary.csv',index=False)
rows=[]
for method,p in methods.items():
    pred=clean if method=='true' else sim_flat(p)
    rows.append(dict(method=method,predicted_vs_observed_curve_RMSE_ms=rmse_flat(pred,obs),predicted_vs_clean_curve_RMSE_ms=rmse_flat(pred,clean)))
pd.DataFrame(rows).to_csv('stage10_5_checked_demo_curve_rmse_summary.csv',index=False)
post_rows=[]
for i,n in enumerate(PARAMS): post_rows.append(dict(parameter=n,true=true_p[i],MAP=map_p[i],posterior_mean=ens['mean'][i],posterior_median=med_p[i],q05=ens['q05'][i],q95=ens['q95'][i],true_inside_90_interval=bool(ens['q05'][i]<=true_p[i]<=ens['q95'][i])))
pd.DataFrame(post_rows).to_csv('stage10_5_checked_ensemble_mcmc_posterior_summary.csv',index=False)
pd.DataFrame([dict(selected_chain_id=int(ens['map_row'].chain_id),selected_step=int(ens['map_row'].step),selection_criterion='maximum_log_posterior',MAP_log_posterior=float(ens['map_row'].log_posterior),MAP_curve_RMSE_ms=float(ens['map_row'].curve_RMSE_ms),MAP_v_top=map_p[0],MAP_v_bottom=map_p[1],MAP_z0=map_p[2],MAP_k=map_p[3])]).to_csv('stage10_5_checked_MAP_selection_by_max_likelihood.csv',index=False)

# plots
plt.figure(figsize=(7,6))
for name,p,ls in [('true',true_p,'-'),('ExtraTrees',reg_p,'--'),('local',loc_p,':'),('MCMC MAP',map_p,'-.'),('posterior median',med_p,'--')]: plt.plot(v_profile(p),ZS,ls,label=name)
plt.gca().invert_yaxis(); plt.xlabel('Velocity, m/s'); plt.ylabel('Depth z, m'); plt.title('Stage 10.5: true vs recovered profile'); plt.grid(True,alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig('stage10_5_checked_profile_true_vs_recovered.png',dpi=200,bbox_inches='tight'); plt.show()

def plot_curves():
    curves={'clean true':clean.reshape(len(SOURCE_Z),NREC),'observed noisy':obs.reshape(len(SOURCE_Z),NREC),'ExtraTrees':sim_flat(reg_p).reshape(len(SOURCE_Z),NREC),'local':sim_flat(loc_p).reshape(len(SOURCE_Z),NREC),'MCMC MAP':sim_flat(map_p).reshape(len(SOURCE_Z),NREC),'posterior median':sim_flat(med_p).reshape(len(SOURCE_Z),NREC)}
    for s,z in enumerate(SOURCE_Z):
        plt.figure(figsize=(9,5))
        for name,C in curves.items(): plt.plot(REC_X,C[s],marker='o',label=name)
        plt.xlabel('Receiver x, m'); plt.ylabel('Arrival time, s'); plt.title(f'Stage 10.5 curves, source_z={int(z)} m'); plt.grid(True,alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig(f'stage10_5_checked_curves_source_z_{int(z)}.png',dpi=200,bbox_inches='tight'); plt.show()
plot_curves()
plt.figure(figsize=(10,5))
for cid,d in ens['all'].groupby('chain_id'): plt.plot(d.step,d.curve_RMSE_ms,label=f'chain {cid}')
plt.xlabel('MCMC step'); plt.ylabel('Curve RMSE, ms'); plt.title('Stage 10.5: convergence of ensemble MCMC'); plt.grid(True,alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig('stage10_5_checked_ensemble_mcmc_convergence_loss.png',dpi=200,bbox_inches='tight'); plt.show()
plt.figure(figsize=(10,5))
for cid,d in ens['all'].groupby('chain_id'): plt.plot(d.step,d.log_posterior,label=f'chain {cid}')
plt.xlabel('MCMC step'); plt.ylabel('log posterior'); plt.title('Stage 10.5: log posterior by chain'); plt.grid(True,alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig('stage10_5_checked_ensemble_mcmc_log_posterior.png',dpi=200,bbox_inches='tight'); plt.show()
for n in PARAMS:
    plt.figure(figsize=(10,5))
    for cid,d in ens['all'].groupby('chain_id'): plt.plot(d.step,d[n],label=f'chain {cid}')
    plt.axhline(true_p[PARAMS.index(n)],ls='--',label='true'); plt.xlabel('MCMC step'); plt.ylabel(n); plt.title(f'Stage 10.5 trace: {n}'); plt.grid(True,alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig(f'stage10_5_checked_ensemble_mcmc_trace_{n}.png',dpi=200,bbox_inches='tight'); plt.show()
for i,n in enumerate(PARAMS):
    plt.figure(figsize=(7,4)); plt.hist(ens['post'][n],bins=30,alpha=.8); plt.axvline(true_p[i],ls='--',label='true'); plt.axvline(map_p[i],ls='-.',label='MAP'); plt.axvline(med_p[i],ls=':',label='median'); plt.xlabel(n); plt.ylabel('count'); plt.title(f'Stage 10.5 posterior: {n}'); plt.grid(True,alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig(f'stage10_5_checked_ensemble_mcmc_posterior_{n}.png',dpi=200,bbox_inches='tight'); plt.show()

def download_stage10_5_checked_results():
    try:
        from google.colab import files
    except Exception:
        files=None
    z=Path('/content/stage10_5_ensemble_mcmc_results.zip')
    fs=sorted(Path('/content/stage10_5_ensemble_mcmc_results').glob('stage10_5_checked_*'))
    fs=[p for p in fs if p.suffix.lower() in ['.csv','.png']]
    print('files found:',len(fs))
    with zipfile.ZipFile(z,'w',zipfile.ZIP_DEFLATED) as zipf:
        for p in fs: zipf.write(p,arcname=p.name)
    print('archive:',z)
    if files: files.download(str(z))

print('\nDONE. To download, run: download_stage10_5_checked_results()')